# Flipkart Order Issue Resolution Dashboard  
## Python Developer Workplace Handover Capstone

You have joined the simulated Flipkart customer support operations engineering team as a **Python Developer**.

This is not a normal classroom assignment. You are taking ownership of an unfinished internal Colab product after the previous developer resigned. The Product Manager has already written a PRD, the Engineering Manager has handed over the broken codebase, and only one core feature is currently working.

**Your final deliverable:** one completed Google Colab notebook with all outputs visible.

## Handover Context

You are expected to behave like the Python Developer who has inherited this notebook.

The business team needs an internal tool that can:

- Load synthetic order operations datasets.
- Detect delayed and risky orders.
- Identify refund-priority cases.
- Summarize customer complaints and support tickets.
- Produce a final issue-resolution report.
- Provide a simple Colab-based search/filter experience.
- Document implementation decisions, assumptions, PRD mapping, and AI prompt usage.

The previous developer completed only the basic `orders.csv` loading feature. The rest of the notebook contains broken code, incomplete logic, fragile assumptions, and TODO sections.

## Documents You Must Read Before Coding

Before completing the notebook, read the documents provided in the student resource folder:

1. `Meeting_Transcript_Flipkart_Order_Issue_Resolution_Dashboard.docx`
2. `PRD_Flipkart_Order_Issue_Resolution_Dashboard.docx`
3. `prd_completion_checklist.csv`
4. `student_readme.md`

Use the PRD as the product source of truth. Use the meeting transcript as the handover context.

## Dataset Upload Instructions for Google Colab

Upload the dataset ZIP file named:

`flipkart_order_issue_resolution_dataset.zip`

The code below extracts the ZIP and creates the dataset path.

The ZIP contains synthetic training data only. It does not contain real Flipkart data.

In [ ]:
from pathlib import Path
import zipfile
import os

DATASET_ZIP = "flipkart_order_issue_resolution_dataset.zip"
EXPECTED_FOLDER = "flipkart_order_issue_resolution_dataset"


try:
    from google.colab import files
    if not Path(DATASET_ZIP).exists():
        print("Upload the dataset ZIP file:", DATASET_ZIP)
        uploaded = files.upload()
except Exception:

    pass

if not Path(DATASET_ZIP).exists():
    raise FileNotFoundError(
        f"{DATASET_ZIP} not found. Upload it to the current Colab session before running this notebook."
    )

if not Path(EXPECTED_FOLDER).exists():
    with zipfile.ZipFile(DATASET_ZIP, "r") as zip_ref:
        zip_ref.extractall(".")

DATA_DIR = Path(EXPECTED_FOLDER)
if not (DATA_DIR / "orders.csv").exists():

    candidates = list(Path(".").rglob("orders.csv"))
    if not candidates:
        raise FileNotFoundError("orders.csv was not found after extracting the dataset ZIP.")
    DATA_DIR = candidates[0].parent

print("Dataset folder ready:", DATA_DIR.resolve())
print("Available files:")
for file in sorted(DATA_DIR.iterdir()):
    print(" -", file.name)

Dataset folder ready: /content/flipkart_order_issue_resolution_dataset
Available files:
 - customer_complaints.json
 - customers.csv
 - orders.csv
 - refunds.xlsx
 - sellers.csv
 - status_notes.txt
 - support_tickets.csv


## Section 1: Imports, Configuration, and Developer Environment Check

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `orders.csv`

**How to approach:**
- Review existing imports.
- Add missing imports only when required.
- Confirm the notebook is running in a Colab-compatible Python environment.

**Expected output format:**
- Printed project name.
- Confirmation that Python/Pandas environment started.

**Hint:**
- Keep imports centralized so later sections are easier to debug.

In [ ]:
# SECTION 1: IMPORTS, CONFIGURATION & ENVIRONMENT CHECK


from pathlib import Path
import os
import sys
import json
import zipfile

import pandas as pd
import numpy as np


PROJECT_NAME = "Flipkart Order Issue Resolution Dashboard"
DATASET_ZIP = "flipkart_order_issue_resolution_dataset.zip"
EXPECTED_FOLDER = "flipkart_order_issue_resolution_dataset"




print(PROJECT_NAME)


print("\nPython environment started successfully.")
print("Python version :", sys.version.split()[0])
print("Pandas version :", pd.__version__)
print("NumPy version  :", np.__version__)


try:
    import google.colab
    RUNNING_IN_COLAB = True
except ImportError:
    RUNNING_IN_COLAB = False

print("Colab environment:", "Detected" if RUNNING_IN_COLAB else "Not detected (local/runtime fallback)")

print("\nRequired core libraries imported successfully.")
print("Environment check completed.")


Flipkart Order Issue Resolution Dashboard

Python environment started successfully.
Python version : 3.13.15
Pandas version : 2.2.3
NumPy version  : 2.1.3
Colab environment: Detected

Required core libraries imported successfully.
Environment check completed.


## Section 2: Load Main Orders Dataset — Working Core Feature

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `orders.csv`

**How to approach:**
- Use the dataset path created during ZIP extraction.
- Load the main order table with Pandas.
- Display the row count and sample rows.

**Expected output format:**
- Printed row count.
- DataFrame preview with order columns.

**Hint:**
- This is the only feature that the previous developer completed reliably.

In [ ]:
# SECTION 2 — LOAD MAIN ORDERS DATASET

orders_file = DATA_DIR / "orders.csv"

if not orders_file.exists():
    raise FileNotFoundError(
        f"orders.csv was not found at: {orders_file}"
    )

orders_df = pd.read_csv(orders_file)

print("Orders loaded successfully.")
print("File:", orders_file)
print("Row count:", len(orders_df))
print("Column count:", len(orders_df.columns))

print("\nOrder columns:")
print(list(orders_df.columns))

print("\nSample order records:")
display(orders_df.head())

print("\nOrders DataFrame information:")
orders_df.info()

Orders loaded successfully.
File: flipkart_order_issue_resolution_dataset/orders.csv
Row count: 5000
Column count: 14

Order columns:
['order_id', 'customer_id', 'seller_id', 'order_date', 'order_time', 'promised_delivery_date', 'actual_delivery_date', 'order_amount', 'payment_method', 'order_status', 'delay_reason', 'delivery_city', 'device_type', 'seller_category']

Sample order records:


,order_id,customer_id,seller_id,order_date,order_time,promised_delivery_date,actual_delivery_date,order_amount,payment_method,order_status,delay_reason,delivery_city,device_type,seller_category
0,ORD0000001,CUST000771,SELL0475,2026-05-30,10:51:46,2026-06-03,2026-06-03,57056.16,Cash on Delivery,Delivered,NaN,Delhi,iOS,Electronics
1,ORD0000002,CUST000122,SELL0035,2026-05-05,15:59:45,2026-05-08,2026-05-07,25038.34,UPI,Delivered,NaN,Gurugram,iOS,Grocery
2,ORD0000003,CUST000784,SELL0785,2026-05-11,11:48:01,2026-05-15,NaN,29439.12,UPI,processing,Warehouse Delay,Lucknow,Android,Grocery
3,ORD0000004,CUST000288,SELL0211,2026-05-23,03:12:27,2026-05-27,2026-05-25,60697.51,Card,Delivered,NaN,Indore,Android,Beauty
4,ORD0000005,CUST001168,SELL0981,2026-06-14,00:30:13,2026-06-19,NaN,56251.50,UPI,Cancelled,Payment Failed,Nagpur,Mobile Web,Beauty



Orders DataFrame information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   order_id                5000 non-null   object 
 1   customer_id             5000 non-null   object 
 2   seller_id               5000 non-null   object 
 3   order_date              5000 non-null   object 
 4   order_time              5000 non-null   object 
 5   promised_delivery_date  4996 non-null   object 
 6   actual_delivery_date    3988 non-null   object 
 7   order_amount            4987 non-null   float64
 8   payment_method          5000 non-null   object 
 9   order_status            5000 non-null   object 
 10  delay_reason            2246 non-null   object 
 11  delivery_city           5000 non-null   object 
 12  device_type             5000 non-null   object 
 13  seller_category         5000 non-null   object 
dtypes: float6

## Section 3: Basic Order Summary

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `orders.csv`

**How to approach:**
- Summarize rows, columns, unique orders, customers, sellers.
- Find duplicate order IDs and missing values.
- Create a status summary.

**Expected output format:**
- One summary table.
- One order status frequency table.

**Hint:**
- The raw column is `order_status`, not `status`.

In [ ]:
# SECTION 3 — BASIC ORDER SUMMARY

summary_table = pd.DataFrame({
    "Metric": [
        "Total Rows",
        "Total Columns",
        "Unique Orders",
        "Unique Customers",
        "Unique Sellers",
        "Duplicate Order IDs",
        "Total Missing Values"
    ],
    "Value": [
        len(orders_df),
        len(orders_df.columns),
        orders_df["order_id"].nunique(),
        orders_df["customer_id"].nunique(),
        orders_df["seller_id"].nunique(),
        orders_df["order_id"].duplicated().sum(),
        orders_df.isna().sum().sum()
    ]
})

print("Basic Order Summary")
display(summary_table)


print("Missing Values by Column")

missing_values = (
    orders_df.isna()
    .sum()
    .reset_index()
)

missing_values.columns = ["Column", "Missing_Count"]

missing_values["Missing_Percentage"] = (
    missing_values["Missing_Count"] / len(orders_df) * 100
).round(2)

display(missing_values)


print("Order Status Frequency")

status_summary = (
    orders_df["order_status"]
    .fillna("Missing")
    .astype(str)
    .str.strip()
    .value_counts()
    .reset_index()
)

status_summary.columns = ["Order_Status", "Frequency"]

status_summary["Percentage"] = (
    status_summary["Frequency"] / len(orders_df) * 100
).round(2)

display(status_summary)

Basic Order Summary


,Metric,Value
0,Total Rows,5000
1,Total Columns,14
2,Unique Orders,4984
3,Unique Customers,1227
4,Unique Sellers,1017
5,Duplicate Order IDs,16
6,Total Missing Values,3783


Missing Values by Column


,Column,Missing_Count,Missing_Percentage
0,order_id,0,0.00
1,customer_id,0,0.00
2,seller_id,0,0.00
3,order_date,0,0.00
4,order_time,0,0.00
5,promised_delivery_date,4,0.08
6,actual_delivery_date,1012,20.24
7,order_amount,13,0.26
8,payment_method,0,0.00
9,order_status,0,0.00


Order Status Frequency


,Order_Status,Frequency,Percentage
0,Delivered,2702,54.04
1,Delayed,583,11.66
2,Cancelled,478,9.56
3,Processing,426,8.52
4,Returned,374,7.48
5,Refund Pending,254,5.08
6,Lost,83,1.66
7,delivered,29,0.58
8,DELIVERED,23,0.46
9,delayed,10,0.20


## Section 4: Debug Fix Log Setup

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `Notebook codebase`

**How to approach:**
- Create a structured debug log.
- Add a helper function to append fixes.
- Update the log after each major repaired section.

**Expected output format:**
- DataFrame with section, issue_found, fix_applied, validation_output.

**Hint:**
- Think like a developer documenting a handover fix, not like a student writing comments.

In [ ]:
# SECTION 4 — DEBUG FIX LOG SETUP

debug_log = []



def add_debug_fix(section, issue_found, fix_applied, validation_output):
    """
    Add one repaired-section entry to the debug log.

    Parameters:
        section (str): Notebook section being repaired.
        issue_found (str): Problem identified in inherited code.
        fix_applied (str): Fix implemented by the developer.
        validation_output (str): Evidence that the fix was validated.
    """

    debug_log.append({
        "section": section,
        "issue_found": issue_found,
        "fix_applied": fix_applied,
        "validation_output": validation_output
    })



def get_debug_log():
    """
    Return the current debug log as a Pandas DataFrame.
    """

    return pd.DataFrame(
        debug_log,
        columns=[
            "section",
            "issue_found",
            "fix_applied",
            "validation_output"
        ]
    )



add_debug_fix(
    section="Section 1",
    issue_found="Imports and project configuration needed to be verified.",
    fix_applied="Reviewed and centralized the required Pandas and NumPy imports.",
    validation_output="Python/Pandas environment started successfully."
)

add_debug_fix(
    section="Section 2",
    issue_found="Inherited core loading section required verification.",
    fix_applied="Loaded orders.csv from the extracted DATA_DIR path.",
    validation_output=f"orders_df loaded successfully with {len(orders_df)} rows."
)

add_debug_fix(
    section="Section 3",
    issue_found="Order status summary referenced incorrect column name 'status'.",
    fix_applied="Changed the status analysis to use the correct 'order_status' column and added operational summary metrics.",
    validation_output="Summary table, duplicate order count, missing-value analysis, and order status frequency table generated successfully."
)



print("Debug fix log initialized successfully.")

debug_log_df = get_debug_log()

display(debug_log_df)

Debug fix log initialized successfully.


,section,issue_found,fix_applied,validation_output
0,Section 1,Imports and project configuration needed to be...,Reviewed and centralized the required Pandas a...,Python/Pandas environment started successfully.
1,Section 2,Inherited core loading section required verifi...,Loaded orders.csv from the extracted DATA_DIR ...,orders_df loaded successfully with 5000 rows.
2,Section 3,Order status summary referenced incorrect colu...,Changed the status analysis to use the correct...,"Summary table, duplicate order count, missing-..."


## Section 5: Multi-File DataLoader

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `orders.csv`
- `customers.csv`
- `sellers.csv`
- `refunds.xlsx`
- `customer_complaints.json`
- `support_tickets.csv`
- `status_notes.txt`

**How to approach:**
- Create a reusable loader class.
- Use the correct method for each file format.
- Add basic exception handling.

**Expected output format:**
- Summary table with dataset name, rows, and columns/text lines.

**Hint:**
- The refunds file is Excel, complaints are JSON, and notes are TXT.

In [ ]:
# SECTION 5 — MULTI-FILE DATA LOADER

from pathlib import Path
import pandas as pd
import json


class FlipkartIssueDataLoader:
    """
    Reusable loader for all Flipkart Order Issue Resolution
    Dashboard input files.
    """

    def __init__(self, folder_path):
        self.folder_path = Path(folder_path)

    def load_dataset_file(self, file_name):
        """
        Load a dataset according to its file extension.

        Supported formats:
        CSV, XLSX/XLS, JSON, TXT
        """

        file_path = self.folder_path / file_name

        if not file_path.exists():
            raise FileNotFoundError(
                f"Dataset file not found: {file_path}"
            )

        try:
            suffix = file_path.suffix.lower()


            if suffix == ".csv":
                return pd.read_csv(file_path)


            elif suffix in [".xlsx", ".xls"]:
                return pd.read_excel(file_path)


            elif suffix == ".json":
                with open(file_path, "r", encoding="utf-8") as file:
                    return json.load(file)


            elif suffix == ".txt":
                with open(file_path, "r", encoding="utf-8") as file:
                    return file.read()

            else:
                raise ValueError(
                    f"Unsupported file format: {suffix}"
                )

        except pd.errors.EmptyDataError:
            raise ValueError(f"Dataset is empty: {file_name}")

        except Exception as e:
            raise RuntimeError(
                f"Error loading {file_name}: {e}"
            )


    def load_all(self):
        """
        Load every required dataset and return them
        in a dictionary.
        """

        data = {}

        required_files = {
            "orders": "orders.csv",
            "customers": "customers.csv",
            "sellers": "sellers.csv",
            "refunds": "refunds.xlsx",
            "complaints": "customer_complaints.json",
            "support_tickets": "support_tickets.csv",
            "status_notes": "status_notes.txt"
        }

        for dataset_name, file_name in required_files.items():

            try:
                data[dataset_name] = self.load_dataset_file(file_name)
                print(f"Loaded successfully: {file_name}")

            except Exception as e:
                print(f"Failed to load {file_name}: {e}")

        return data


loader = FlipkartIssueDataLoader(DATA_DIR)

all_data = loader.load_all()


orders_df = all_data.get("orders")
customers_df = all_data.get("customers")
sellers_df = all_data.get("sellers")
refunds_df = all_data.get("refunds")
complaints_data = all_data.get("complaints")
support_tickets_df = all_data.get("support_tickets")
status_notes = all_data.get("status_notes")



dataset_summary = []

for dataset_name, dataset in all_data.items():

    if isinstance(dataset, pd.DataFrame):

        dataset_summary.append({
            "Dataset": dataset_name,
            "Source_File": "Loaded DataFrame",
            "Rows": len(dataset),
            "Columns": len(dataset.columns),
            "Text_Lines": None
        })

    elif isinstance(dataset, dict):
        dataset_summary.append({
            "Dataset": dataset_name,
            "Source_File": "JSON",
            "Rows": len(dataset) if isinstance(dataset, list) else None,
            "Columns": None,
            "Text_Lines": None
        })

    elif isinstance(dataset, list):
        dataset_summary.append({
            "Dataset": dataset_name,
            "Source_File": "JSON",
            "Rows": len(dataset),
            "Columns": None,
            "Text_Lines": None
        })

    elif isinstance(dataset, str):

        dataset_summary.append({
            "Dataset": dataset_name,
            "Source_File": "TXT",
            "Rows": None,
            "Columns": None,
            "Text_Lines": len(dataset.splitlines())
        })


dataset_summary_df = pd.DataFrame(dataset_summary)


print("\nAll required datasets processed.")

print("\nLoaded dataset summary:")
display(dataset_summary_df)

print("\nAvailable datasets:")
print(list(all_data.keys()))


add_debug_fix(
    section="Section 5",
    issue_found="Customer and seller filenames were incorrect, refunds were loaded with the wrong CSV method, and JSON/TXT files were not loaded.",
    fix_applied="Implemented a reusable FlipkartIssueDataLoader with CSV, Excel, JSON, and TXT support plus exception handling.",
    validation_output=f"Loaded datasets: {list(all_data.keys())}"
)

display(get_debug_log())


Loaded successfully: orders.csv
Loaded successfully: customers.csv
Loaded successfully: sellers.csv
Loaded successfully: refunds.xlsx
Loaded successfully: customer_complaints.json
Loaded successfully: support_tickets.csv
Loaded successfully: status_notes.txt

All required datasets processed.

Loaded dataset summary:


,Dataset,Source_File,Rows,Columns,Text_Lines
0,orders,Loaded DataFrame,5000.0,14.0,NaN
1,customers,Loaded DataFrame,1200.0,6.0,NaN
2,sellers,Loaded DataFrame,1000.0,6.0,NaN
3,refunds,Loaded DataFrame,1300.0,8.0,NaN
4,complaints,JSON,1500.0,NaN,NaN
5,support_tickets,Loaded DataFrame,1500.0,8.0,NaN
6,status_notes,TXT,NaN,NaN,100.0



Available datasets:
['orders', 'customers', 'sellers', 'refunds', 'complaints', 'support_tickets', 'status_notes']


,section,issue_found,fix_applied,validation_output
0,Section 1,Imports and project configuration needed to be...,Reviewed and centralized the required Pandas a...,Python/Pandas environment started successfully.
1,Section 2,Inherited core loading section required verifi...,Loaded orders.csv from the extracted DATA_DIR ...,orders_df loaded successfully with 5000 rows.
2,Section 3,Order status summary referenced incorrect colu...,Changed the status analysis to use the correct...,"Summary table, duplicate order count, missing-..."
3,Section 5,"Customer and seller filenames were incorrect, ...",Implemented a reusable FlipkartIssueDataLoader...,"Loaded datasets: ['orders', 'customers', 'sell..."


## Section 6: Data Validation and Relationship Checks

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `orders.csv`
- `customers.csv`
- `sellers.csv`
- `refunds.xlsx`
- `customer_complaints.json`
- `support_tickets.csv`

**How to approach:**
- Check required columns.
- Check duplicate order IDs.
- Check customer/seller/order relationships.
- Flag invalid dates and suspicious amounts.

**Expected output format:**
- Validation summary table with check_name, status, issue_count, notes.

**Hint:**
- Do not delete invalid records silently; flag them as review items.

In [ ]:
# SECTION 6: DATA VALIDATION AND RELATIONSHIP CHECKS

import pandas as pd
import numpy as np

validation_results = []
review_checks = []


def add_validation(check_name, status, issue_count, notes):
    validation_results.append({
        "check_name": check_name,
        "status": status,
        "issue_count": int(issue_count),
        "notes": notes
    })



complaints_validation_df = pd.DataFrame(
    all_data.get("complaints", [])
).copy()

required_columns = {
    "orders": [
        "order_id",
        "customer_id",
        "seller_id",
        "order_amount",
        "order_status"
    ],
    "customers": [
        "customer_id"
    ],
    "sellers": [
        "seller_id"
    ],
    "refunds": [
        "refund_id",
        "order_id",
        "refund_status",
        "refund_amount"
    ],
    "complaints": [
        "complaint_id",
        "order_id",
        "customer_id"
    ],
    "support_tickets": [
        "ticket_id",
        "order_id",
        "customer_id"
    ]
}

dataset_frames = {
    "orders": all_data.get("orders", pd.DataFrame()),
    "customers": all_data.get("customers", pd.DataFrame()),
    "sellers": all_data.get("sellers", pd.DataFrame()),
    "refunds": all_data.get("refunds", pd.DataFrame()),
    "complaints": complaints_validation_df,
    "support_tickets": all_data.get("support_tickets", pd.DataFrame())
}

for dataset_name, required_cols in required_columns.items():

    df = dataset_frames[dataset_name]

    missing_cols = [
        col for col in required_cols
        if col not in df.columns
    ]

    if len(missing_cols) == 0:
        add_validation(
            f"{dataset_name} required columns",
            "PASS",
            0,
            "All required columns are present."
        )
    else:
        add_validation(
            f"{dataset_name} required columns",
            "REVIEW",
            len(missing_cols),
            f"Missing columns: {missing_cols}"
        )
        review_checks.append(f"{dataset_name}: missing required columns")


if "order_id" in orders_df.columns:

    duplicate_order_ids = (
        orders_df["order_id"]
        .astype(str)
        .duplicated(keep=False)
    )

    duplicate_count = int(duplicate_order_ids.sum())

    if duplicate_count == 0:
        add_validation(
            "Duplicate order IDs",
            "PASS",
            0,
            "No duplicate order IDs found."
        )
    else:
        add_validation(
            "Duplicate order IDs",
            "REVIEW",
            duplicate_count,
            "Duplicate order IDs were found and flagged for review."
        )
        review_checks.append("Duplicate order IDs")


if (
    "customer_id" in orders_df.columns
    and "customer_id" in customers_df.columns
):

    valid_customers = set(
        customers_df["customer_id"]
        .dropna()
        .astype(str)
    )

    orphan_customer_mask = (
        orders_df["customer_id"]
        .notna()
        & ~orders_df["customer_id"].astype(str).isin(valid_customers)
    )

    orphan_customer_count = int(orphan_customer_mask.sum())

    if orphan_customer_count == 0:
        add_validation(
            "Order-Customer relationship",
            "PASS",
            0,
            "All order customer IDs have matching customer records."
        )
    else:
        add_validation(
            "Order-Customer relationship",
            "REVIEW",
            orphan_customer_count,
            "Orders contain customer IDs not found in customers data."
        )
        review_checks.append("Orphan customer references")


if (
    "seller_id" in orders_df.columns
    and "seller_id" in sellers_df.columns
):

    valid_sellers = set(
        sellers_df["seller_id"]
        .dropna()
        .astype(str)
    )

    orphan_seller_mask = (
        orders_df["seller_id"]
        .notna()
        & ~orders_df["seller_id"].astype(str).isin(valid_sellers)
    )

    orphan_seller_count = int(orphan_seller_mask.sum())

    if orphan_seller_count == 0:
        add_validation(
            "Order-Seller relationship",
            "PASS",
            0,
            "All order seller IDs have matching seller records."
        )
    else:
        add_validation(
            "Order-Seller relationship",
            "REVIEW",
            orphan_seller_count,
            "Orders contain seller IDs not found in sellers data."
        )
        review_checks.append("Orphan seller references")


if (
    "order_id" in refunds_df.columns
    and "order_id" in orders_df.columns
):

    valid_order_ids = set(
        orders_df["order_id"]
        .dropna()
        .astype(str)
    )

    orphan_refund_mask = (
        refunds_df["order_id"]
        .notna()
        & ~refunds_df["order_id"].astype(str).isin(valid_order_ids)
    )

    orphan_refund_count = int(orphan_refund_mask.sum())

    if orphan_refund_count == 0:
        add_validation(
            "Refund-Order relationship",
            "PASS",
            0,
            "All refund records are linked to valid orders."
        )
    else:
        add_validation(
            "Refund-Order relationship",
            "REVIEW",
            orphan_refund_count,
            "Refund records contain order IDs not found in orders data."
        )
        review_checks.append("Orphan refund references")


if (
    "order_id" in complaints_validation_df.columns
    and "order_id" in orders_df.columns
):

    orphan_complaint_mask = (
        complaints_validation_df["order_id"]
        .notna()
        & ~complaints_validation_df["order_id"].astype(str).isin(valid_order_ids)
    )

    orphan_complaint_count = int(orphan_complaint_mask.sum())

    if orphan_complaint_count == 0:
        add_validation(
            "Complaint-Order relationship",
            "PASS",
            0,
            "All complaint order IDs have matching orders."
        )
    else:
        add_validation(
            "Complaint-Order relationship",
            "REVIEW",
            orphan_complaint_count,
            "Complaints contain order IDs not found in orders data."
        )
        review_checks.append("Orphan complaint references")


if (
    "order_id" in support_tickets_df.columns
    and "order_id" in orders_df.columns
):

    orphan_ticket_mask = (
        support_tickets_df["order_id"]
        .notna()
        & ~support_tickets_df["order_id"].astype(str).isin(valid_order_ids)
    )

    orphan_ticket_count = int(orphan_ticket_mask.sum())

    if orphan_ticket_count == 0:
        add_validation(
            "Support Ticket-Order relationship",
            "PASS",
            0,
            "All support ticket order IDs have matching orders."
        )
    else:
        add_validation(
            "Support Ticket-Order relationship",
            "REVIEW",
            orphan_ticket_count,
            "Support tickets contain order IDs not found in orders data."
        )
        review_checks.append("Orphan support ticket references")


date_columns = [
    "order_date",
    "promised_delivery_date",
    "actual_delivery_date"
]

invalid_date_count = 0
invalid_date_details = []

for col in date_columns:

    if col in orders_df.columns:

        parsed_dates = pd.to_datetime(
            orders_df[col],
            errors="coerce"
        )

        invalid_mask = (
            orders_df[col].notna()
            & parsed_dates.isna()
        )

        count = int(invalid_mask.sum())

        if count > 0:
            invalid_date_count += count
            invalid_date_details.append(
                f"{col}: {count}"
            )

if invalid_date_count == 0:

    add_validation(
        "Invalid order dates",
        "PASS",
        0,
        "No invalid order date values detected."
    )

else:

    add_validation(
        "Invalid order dates",
        "REVIEW",
        invalid_date_count,
        "Invalid dates found: " + ", ".join(invalid_date_details)
    )

    review_checks.append("Invalid order dates")


invalid_date_ordering_count = 0

if (
    "order_date" in orders_df.columns
    and "promised_delivery_date" in orders_df.columns
):

    temp_order_date = pd.to_datetime(
        orders_df["order_date"],
        errors="coerce"
    )

    temp_promised_date = pd.to_datetime(
        orders_df["promised_delivery_date"],
        errors="coerce"
    )

    invalid_ordering_mask = (
        temp_order_date.notna()
        & temp_promised_date.notna()
        & (temp_promised_date < temp_order_date)
    )

    invalid_date_ordering_count += int(
        invalid_ordering_mask.sum()
    )


if (
    "order_date" in orders_df.columns
    and "actual_delivery_date" in orders_df.columns
):

    temp_order_date = pd.to_datetime(
        orders_df["order_date"],
        errors="coerce"
    )

    temp_actual_date = pd.to_datetime(
        orders_df["actual_delivery_date"],
        errors="coerce"
    )

    invalid_ordering_mask = (
        temp_order_date.notna()
        & temp_actual_date.notna()
        & (temp_actual_date < temp_order_date)
    )

    invalid_date_ordering_count += int(
        invalid_ordering_mask.sum()
    )


if invalid_date_ordering_count == 0:

    add_validation(
        "Date consistency",
        "PASS",
        0,
        "No impossible delivery date ordering detected."
    )

else:

    add_validation(
        "Date consistency",
        "REVIEW",
        invalid_date_ordering_count,
        "Some dates occur before the order date."
    )

    review_checks.append("Invalid date ordering")



order_amount_review_count = 0

if "order_amount" in orders_df.columns:

    order_amount_numeric = pd.to_numeric(
        orders_df["order_amount"],
        errors="coerce"
    )

    negative_amount_mask = order_amount_numeric < 0
    zero_amount_mask = order_amount_numeric == 0

    order_amount_review_mask = (
        negative_amount_mask
        | zero_amount_mask
        | order_amount_numeric.isna()
    )

    order_amount_review_count = int(
        order_amount_review_mask.sum()
    )

if order_amount_review_count == 0:

    add_validation(
        "Suspicious order amounts",
        "PASS",
        0,
        "No negative, zero, or non-numeric order amounts found."
    )

else:

    add_validation(
        "Suspicious order amounts",
        "REVIEW",
        order_amount_review_count,
        "Negative, zero, or invalid order amounts were flagged."
    )

    review_checks.append("Suspicious order amounts")

refund_amount_review_count = 0

if (
    "refund_amount" in refunds_df.columns
    and "order_id" in refunds_df.columns
    and "order_id" in orders_df.columns
    and "order_amount" in orders_df.columns
):

    refund_amount_numeric = pd.to_numeric(
        refunds_df["refund_amount"],
        errors="coerce"
    )

    negative_refund_mask = refund_amount_numeric < 0
    invalid_refund_mask = refund_amount_numeric.isna()

    refund_with_orders = refunds_df[
        refunds_df["order_id"].astype(str).isin(valid_order_ids)
    ].copy()

    order_amount_lookup = (
        orders_df[["order_id", "order_amount"]]
        .drop_duplicates("order_id")
        .copy()
    )

    order_amount_lookup["order_id"] = (
        order_amount_lookup["order_id"].astype(str)
    )

    order_amount_lookup["order_amount"] = pd.to_numeric(
        order_amount_lookup["order_amount"],
        errors="coerce"
    )

    refund_with_orders["order_id"] = (
        refund_with_orders["order_id"].astype(str)
    )

    refund_with_orders = refund_with_orders.merge(
        order_amount_lookup,
        on="order_id",
        how="left"
    )

    suspicious_refund_mask = (
        refund_with_orders["refund_amount"].isna()
        | (pd.to_numeric(
            refund_with_orders["refund_amount"],
            errors="coerce"
        ) < 0)
        | (
            pd.to_numeric(
                refund_with_orders["refund_amount"],
                errors="coerce"
            )
            > refund_with_orders["order_amount"]
        )
    )

    refund_amount_review_count = (
        int(negative_refund_mask.sum())
        + int(invalid_refund_mask.sum())
        + int(suspicious_refund_mask.sum())
    )


if refund_amount_review_count == 0:

    add_validation(
        "Suspicious refund amounts",
        "PASS",
        0,
        "No negative, invalid, or order-value-exceeding refund amounts found."
    )

else:

    add_validation(
        "Suspicious refund amounts",
        "REVIEW",
        refund_amount_review_count,
        "Suspicious refund amounts were flagged for review."
    )

    review_checks.append("Suspicious refund amounts")

validation_summary_df = pd.DataFrame(
    validation_results,
    columns=[
        "check_name",
        "status",
        "issue_count",
        "notes"
    ]
)

print("VALIDATION SUMMARY")
display(validation_summary_df)

print("\nReview items found:", len(review_checks))

if len(review_checks) > 0:
    print("Items requiring review:")
    for item in review_checks:
        print("-", item)
else:
    print("No review items found.")


add_debug_fix(
    section="Section 6",
    issue_found=(
        "The inherited validation logic used incorrect dataset field names "
        "and did not comprehensively check required columns, duplicate "
        "order IDs, customer/seller relationships, orphan records, "
        "invalid dates, date consistency, and suspicious amounts."
    ),
    fix_applied=(
        "Corrected the validation fields to match the loaded datasets "
        "and implemented required-column checks, duplicate order checks, "
        "customer/seller/order relationship checks, orphan-link checks, "
        "invalid date checks, date consistency checks, and suspicious "
        "order/refund amount checks. Invalid records are flagged for "
        "review instead of being silently deleted."
    ),
    validation_output=(
        f"{len(validation_summary_df)} validation checks completed; "
        f"{len(review_checks)} checks require review."
    )
)

display(get_debug_log())

VALIDATION SUMMARY


,check_name,status,issue_count,notes
0,orders required columns,PASS,0,All required columns are present.
1,customers required columns,PASS,0,All required columns are present.
2,sellers required columns,PASS,0,All required columns are present.
3,refunds required columns,PASS,0,All required columns are present.
4,complaints required columns,PASS,0,All required columns are present.
5,support_tickets required columns,PASS,0,All required columns are present.
6,Duplicate order IDs,REVIEW,32,Duplicate order IDs were found and flagged for...
7,Order-Customer relationship,REVIEW,40,Orders contain customer IDs not found in custo...
8,Order-Seller relationship,REVIEW,30,Orders contain seller IDs not found in sellers...
9,Refund-Order relationship,REVIEW,23,Refund records contain order IDs not found in ...



Review items found: 9
Items requiring review:
- Duplicate order IDs
- Orphan customer references
- Orphan seller references
- Orphan refund references
- Orphan complaint references
- Orphan support ticket references
- Invalid order dates
- Suspicious order amounts
- Suspicious refund amounts


,section,issue_found,fix_applied,validation_output
0,Section 1,Imports and project configuration needed to be...,Reviewed and centralized the required Pandas a...,Python/Pandas environment started successfully.
1,Section 2,Inherited core loading section required verifi...,Loaded orders.csv from the extracted DATA_DIR ...,orders_df loaded successfully with 5000 rows.
2,Section 3,Order status summary referenced incorrect colu...,Changed the status analysis to use the correct...,"Summary table, duplicate order count, missing-..."
3,Section 5,"Customer and seller filenames were incorrect, ...",Implemented a reusable FlipkartIssueDataLoader...,"Loaded datasets: ['orders', 'customers', 'sell..."
4,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
5,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
6,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...


## Section 7: Data Cleaning and Standardization

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `All loaded datasets`

**How to approach:**
- Standardize text statuses.
- Trim whitespace and normalize casing.
- Parse dates using safe coercion.
- Convert amount and time columns to numeric.

**Expected output format:**
- Cleaning summary table.
- Cleaned DataFrame preview.

**Hint:**
- Use helper functions so cleaning logic is not repeated across every dataset.

In [ ]:
# SECTION 7: DATA CLEANING AND STANDARDIZATION

import pandas as pd
import numpy as np



def standardize_text_columns(df):
    """
    Trim whitespace, normalize casing, and clean separators
    for all object/string columns.
    """
    df = df.copy()

    for col in df.select_dtypes(include=["object", "string"]).columns:


        df[col] = df[col].astype("string")


        df[col] = df[col].str.strip()


        df[col] = df[col].str.replace(
            r"\s+",
            " ",
            regex=True
        )


        df[col] = df[col].str.replace(
            r"[_\-]+",
            "_",
            regex=True
        )


        df[col] = df[col].str.lower()

    return df


def standardize_date_columns(df, date_columns):
    """
    Convert date columns using safe coercion.
    Invalid dates become NaT instead of causing an error.
    """
    df = df.copy()

    for col in date_columns:
        if col in df.columns:
            df[col] = pd.to_datetime(
                df[col],
                errors="coerce"
            )

    return df


def standardize_numeric_columns(df, numeric_columns):
    """
    Convert amount/time columns to numeric.
    Invalid values become NaN.
    """
    df = df.copy()

    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col],
                errors="coerce"
            )

    return df


orders_clean = standardize_text_columns(orders_df)
customers_clean = standardize_text_columns(customers_df)
sellers_clean = standardize_text_columns(sellers_df)
refunds_clean = standardize_text_columns(refunds_df)
support_tickets_clean = standardize_text_columns(support_tickets_df)


complaints_clean = standardize_text_columns(
    complaints_validation_df
)


orders_clean = standardize_date_columns(
    orders_clean,
    [
        "order_date",
        "promised_delivery_date",
        "actual_delivery_date"
    ]
)


refunds_clean = standardize_date_columns(
    refunds_clean,
    [
        "refund_initiated_date",
        "refund_completed_date"
    ]
)


complaints_clean = standardize_date_columns(
    complaints_clean,
    [
        "complaint_date",
        "created_at",
        "updated_at"
    ]
)


support_tickets_clean = standardize_date_columns(
    support_tickets_clean,
    [
        "ticket_date",
        "created_at",
        "updated_at",
        "closed_at",
        "resolved_at"
    ]
)


orders_clean = standardize_numeric_columns(
    orders_clean,
    [
        "order_amount"
    ]
)

refunds_clean = standardize_numeric_columns(
    refunds_clean,
    [
        "refund_amount"
    ]
)

support_tickets_clean = standardize_numeric_columns(
    support_tickets_clean,
    [
        "resolution_time_hours"
    ]
)

def create_cleaning_summary(
    original_df,
    cleaned_df,
    dataset_name
):
    """
    Create a compact summary showing rows, columns,
    missing values and date/numeric conversion effects.
    """

    summary = {
        "dataset": dataset_name,
        "rows_before": len(original_df),
        "rows_after": len(cleaned_df),
        "columns": len(cleaned_df.columns),
        "missing_values_before": int(
            original_df.isna().sum().sum()
        ),
        "missing_values_after": int(
            cleaned_df.isna().sum().sum()
        ),
        "status": "Cleaned"
    }

    return summary


cleaning_summary = []

cleaning_summary.append(
    create_cleaning_summary(
        orders_df,
        orders_clean,
        "orders"
    )
)

cleaning_summary.append(
    create_cleaning_summary(
        customers_df,
        customers_clean,
        "customers"
    )
)

cleaning_summary.append(
    create_cleaning_summary(
        sellers_df,
        sellers_clean,
        "sellers"
    )
)

cleaning_summary.append(
    create_cleaning_summary(
        refunds_df,
        refunds_clean,
        "refunds"
    )
)

cleaning_summary.append(
    create_cleaning_summary(
        complaints_validation_df,
        complaints_clean,
        "complaints"
    )
)

cleaning_summary.append(
    create_cleaning_summary(
        support_tickets_df,
        support_tickets_clean,
        "support_tickets"
    )
)

cleaning_summary_df = pd.DataFrame(
    cleaning_summary
)



print("DATA CLEANING SUMMARY")
display(cleaning_summary_df)


print("\nORDERS CLEANED PREVIEW")
display(orders_clean.head())

print("\nCUSTOMERS CLEANED PREVIEW")
display(customers_clean.head())

print("\nSELLERS CLEANED PREVIEW")
display(sellers_clean.head())

print("\nREFUNDS CLEANED PREVIEW")
display(refunds_clean.head())

print("\nCOMPLAINTS CLEANED PREVIEW")
display(complaints_clean.head())

print("\nSUPPORT TICKETS CLEANED PREVIEW")
display(support_tickets_clean.head())


print("\nPOST-CLEANING DATA TYPES")

print("\nOrders:")
print(
    orders_clean[
        [
            col for col in [
                "order_amount",
                "order_date",
                "promised_delivery_date",
                "actual_delivery_date"
            ]
            if col in orders_clean.columns
        ]
    ].dtypes
)

print("\nRefunds:")
print(
    refunds_clean[
        [
            col for col in [
                "refund_amount",
                "refund_initiated_date",
                "refund_completed_date"
            ]
            if col in refunds_clean.columns
        ]
    ].dtypes
)

print("\nSupport Tickets:")
print(
    support_tickets_clean[
        [
            col for col in [
                "resolution_time_hours"
            ]
            if col in support_tickets_clean.columns
        ]
    ].dtypes
)



add_debug_fix(
    section="Section 7",
    issue_found=(
        "The inherited notebook repeated cleaning logic and did not "
        "consistently standardize text fields, parse dates safely, "
        "or convert amount/time fields to numeric values."
    ),
    fix_applied=(
        "Created reusable helper functions for text, date, and numeric "
        "standardization and applied them across all loaded datasets. "
        "Dates use safe coercion and invalid numeric values are converted "
        "to NaN for review instead of causing runtime failures."
    ),
    validation_output=(
        f"{len(cleaning_summary_df)} datasets cleaned successfully; "
        "cleaned previews and resulting data types were displayed."
    )
)

display(get_debug_log())

DATA CLEANING SUMMARY


/tmp/ipykernel_2252/436445031.py:52: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(
/tmp/ipykernel_2252/436445031.py:52: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(
/tmp/ipykernel_2252/436445031.py:52: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(
/tmp/ipykernel_2252/436445031.py:52: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(
/tmp

,dataset,rows_before,rows_after,columns,missing_values_before,missing_values_after,status
0,orders,5000,5000,14,3783,17767,Cleaned
1,customers,1200,1200,6,0,0,Cleaned
2,sellers,1000,1000,6,0,0,Cleaned
3,refunds,1300,1300,8,599,2600,Cleaned
4,complaints,1500,1500,8,0,1500,Cleaned
5,support_tickets,1500,1500,8,151,151,Cleaned



ORDERS CLEANED PREVIEW


,order_id,customer_id,seller_id,order_date,order_time,promised_delivery_date,actual_delivery_date,order_amount,payment_method,order_status,delay_reason,delivery_city,device_type,seller_category
0,ord0000001,cust000771,sell0475,NaT,10:51:46,NaT,NaT,57056.16,cash on delivery,delivered,<NA>,delhi,ios,electronics
1,ord0000002,cust000122,sell0035,NaT,15:59:45,NaT,NaT,25038.34,upi,delivered,<NA>,gurugram,ios,grocery
2,ord0000003,cust000784,sell0785,NaT,11:48:01,NaT,NaT,29439.12,upi,processing,warehouse delay,lucknow,android,grocery
3,ord0000004,cust000288,sell0211,NaT,03:12:27,NaT,NaT,60697.51,card,delivered,<NA>,indore,android,beauty
4,ord0000005,cust001168,sell0981,NaT,00:30:13,NaT,NaT,56251.50,upi,cancelled,payment failed,nagpur,mobile web,beauty


...

,complaint_id,order_id,customer_id,complaint_date,complaint_type,complaint_description,complaint_status,sentiment_tag
0,cmp000001,ord0003983,cust000684,NaT,damaged product,customer reported damaged product for order or...,resolved,neutral
1,cmp000002,ord0001009,cust000185,NaT,wrong item,customer reported wrong item for order ord0001...,resolved,neutral
2,cmp000003,ord0003634,cust000082,NaT,missing item,customer reported missing item for order ord00...,resolved,angry
3,cmp000004,ord0001154,cust000565,NaT,wrong item,customer reported wrong item for order ord0001...,open,negative
4,cmp000005,ord0003080,cust000493,NaT,cancellation issue,customer reported cancellation issue for order...,escalated,neutral



SUPPORT TICKETS CLEANED PREVIEW


,ticket_id,order_id,customer_id,ticket_created_date,ticket_status,assigned_team,resolution_time_hours,escalation_flag
0,tkt000001,ord0000759,cust001184,2026_06_17,open,logistics ops,82.0,yes
1,tkt000002,ord0004152,custx954637,2026_05_08,open,logistics ops,92.0,no
2,tkt000003,ord0000190,cust000698,2026_06_16,pending,escalation desk,176.0,yes
3,tkt000004,ord0003682,cust000079,2026_06_10,resolved,support ops,128.0,no
4,tkt000005,ord0003794,cust000939,2026_06_01,resolved,escalation desk,148.0,yes



POST-CLEANING DATA TYPES

Orders:
order_amount                     float64
order_date                datetime64[ns]
promised_delivery_date    datetime64[ns]
actual_delivery_date      datetime64[ns]
dtype: object

Refunds:
refund_amount                   float64
refund_initiated_date    datetime64[ns]
refund_completed_date    datetime64[ns]
dtype: object

Support Tickets:
resolution_time_hours    float64
dtype: object


,section,issue_found,fix_applied,validation_output
0,Section 1,Imports and project configuration needed to be...,Reviewed and centralized the required Pandas a...,Python/Pandas environment started successfully.
1,Section 2,Inherited core loading section required verifi...,Loaded orders.csv from the extracted DATA_DIR ...,orders_df loaded successfully with 5000 rows.
2,Section 3,Order status summary referenced incorrect colu...,Changed the status analysis to use the correct...,"Summary table, duplicate order count, missing-..."
3,Section 5,"Customer and seller filenames were incorrect, ...",Implemented a reusable FlipkartIssueDataLoader...,"Loaded datasets: ['orders', 'customers', 'sell..."
4,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
5,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
6,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
7,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
8,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
9,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...


## Section 8: Order Delay Detection

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `orders.csv`

**How to approach:**
- Compare promised and actual delivery dates.
- Handle pending orders that breached promise date.
- Classify delay buckets.
- Summarize delays by bucket and city.

**Expected output format:**
- delay_summary DataFrame.
- city_delay_summary DataFrame.
- orders table with delivery_delay_days and delay_bucket.

**Hint:**
- Use `actual_delivery_date`, not `delivered_date`.

In [ ]:
# SECTION 8 — ORDER DELAY DETECTION

from datetime import datetime

orders_delay = orders_clean.copy()


orders_delay["promised_delivery_date"] = pd.to_datetime(
    orders_delay["promised_delivery_date"],
    errors="coerce"
)

orders_delay["actual_delivery_date"] = pd.to_datetime(
    orders_delay["actual_delivery_date"],
    errors="coerce"
)

orders_delay["order_status_clean"] = (
    orders_delay["order_status"]
    .astype("string")
    .str.strip()
    .str.lower()
)

today = pd.Timestamp.today().normalize()

orders_delay["delivery_delay_days"] = np.nan


actual_date_mask = (
    orders_delay["actual_delivery_date"].notna() &
    orders_delay["promised_delivery_date"].notna()
)

orders_delay.loc[actual_date_mask, "delivery_delay_days"] = (
    orders_delay.loc[actual_date_mask, "actual_delivery_date"] -
    orders_delay.loc[actual_date_mask, "promised_delivery_date"]
).dt.days


pending_breach_mask = (
    orders_delay["actual_delivery_date"].isna() &
    orders_delay["promised_delivery_date"].notna() &
    (orders_delay["order_status_clean"].isin([
        "pending",
        "processing",
        "shipped",
        "in transit",
        "in_transit",
        "out for delivery",
        "out_for_delivery"
    ]))
)

orders_delay.loc[pending_breach_mask, "delivery_delay_days"] = (
    today -
    orders_delay.loc[pending_breach_mask, "promised_delivery_date"]
).dt.days


def classify_delay_bucket(delay_days, promised_date, actual_date):
    """
    Classify an order into the PRD delay buckets.
    """


    if pd.isna(promised_date):
        return "Delivery Date Missing or Invalid"


    if pd.isna(actual_date) and pd.isna(delay_days):
        return "Delivery Date Missing or Invalid"


    if delay_days <= 0:
        return "On Time"


    elif delay_days <= 2:
        return "Minor Delay"


    elif delay_days <= 7:
        return "Moderate Delay"


    else:
        return "Severe Delay"


orders_delay["delay_bucket"] = orders_delay.apply(
    lambda row: classify_delay_bucket(
        row["delivery_delay_days"],
        row["promised_delivery_date"],
        row["actual_delivery_date"]
    ),
    axis=1
)

delay_summary = (
    orders_delay["delay_bucket"]
    .value_counts()
    .rename_axis("delay_bucket")
    .reset_index(name="order_count")
)

delay_summary["percentage"] = (
    delay_summary["order_count"] /
    len(orders_delay) *
    100
).round(2)

city_column = "delivery_city"

if city_column not in orders_delay.columns:

    if "city" in orders_delay.columns:
        city_column = "city"

city_delay_summary = (
    orders_delay
    .groupby(city_column, dropna=False)
    .agg(
        total_orders=("order_id", "nunique"),
        average_delay_days=("delivery_delay_days", "mean"),
        delayed_orders=(
            "delivery_delay_days",
            lambda x: (x > 0).sum()
        ),
        severe_delay_orders=(
            "delay_bucket",
            lambda x: (x == "Severe Delay").sum()
        )
    )
    .reset_index()
)

city_delay_summary["average_delay_days"] = (
    city_delay_summary["average_delay_days"]
    .round(2)
)

city_delay_summary["delay_rate_percent"] = (
    city_delay_summary["delayed_orders"] /
    city_delay_summary["total_orders"] *
    100
).round(2)

city_delay_summary = city_delay_summary.sort_values(
    by=["delayed_orders", "average_delay_days"],
    ascending=False
)

orders_clean = orders_delay.copy()

print("ORDER DELAY SUMMARY")

display(delay_summary)


print("\nCITY-LEVEL DELAY SUMMARY")
display(city_delay_summary)


print("\nORDERS WITH DELAY FEATURES")
display(
    orders_clean[
        [
            "order_id",
            "order_status_clean",
            "promised_delivery_date",
            "actual_delivery_date",
            "delivery_delay_days",
            "delay_bucket"
        ]
    ].head(10)
)


add_debug_fix(
    section="Section 8",
    issue_found=(
        "Inherited delay logic referenced non-existent delivered_date "
        "and did not handle pending orders with breached promised dates."
    ),
    fix_applied=(
        "Changed the calculation to actual_delivery_date, added pending "
        "promise-date breach logic, safe date handling, delay buckets, "
        "and city-level delay summaries."
    ),
    validation_output=(
        f"Delay analysis completed for {len(orders_clean)} orders; "
        f"{len(delay_summary)} delay buckets generated."
    )
)

display(get_debug_log())

ORDER DELAY SUMMARY


,delay_bucket,order_count,percentage
0,Delivery Date Missing or Invalid,5000,100.0



CITY-LEVEL DELAY SUMMARY


,delivery_city,total_orders,average_delay_days,delayed_orders,severe_delay_orders,delay_rate_percent
0,ahmedabad,306,NaN,0,0,0.0
1,bengaluru,282,NaN,0,0,0.0
2,chennai,344,NaN,0,0,0.0
3,coimbatore,302,NaN,0,0,0.0
4,delhi,327,NaN,0,0,0.0
5,gurugram,321,NaN,0,0,0.0
6,hyderabad,294,NaN,0,0,0.0
7,indore,309,NaN,0,0,0.0
8,jaipur,323,NaN,0,0,0.0
9,kochi,313,NaN,0,0,0.0



ORDERS WITH DELAY FEATURES


,order_id,order_status_clean,promised_delivery_date,actual_delivery_date,delivery_delay_days,delay_bucket
0,ord0000001,delivered,NaT,NaT,NaN,Delivery Date Missing or Invalid
1,ord0000002,delivered,NaT,NaT,NaN,Delivery Date Missing or Invalid
2,ord0000003,processing,NaT,NaT,NaN,Delivery Date Missing or Invalid
3,ord0000004,delivered,NaT,NaT,NaN,Delivery Date Missing or Invalid
4,ord0000005,cancelled,NaT,NaT,NaN,Delivery Date Missing or Invalid
5,ord0000006,delivered,NaT,NaT,NaN,Delivery Date Missing or Invalid
6,ord0000007,delivered,NaT,NaT,NaN,Delivery Date Missing or Invalid
7,ord0000008,cancelled,NaT,NaT,NaN,Delivery Date Missing or Invalid
8,ord0000009,refund pending,NaT,NaT,NaN,Delivery Date Missing or Invalid
9,ord0000010,cancelled,NaT,NaT,NaN,Delivery Date Missing or Invalid


,section,issue_found,fix_applied,validation_output
0,Section 1,Imports and project configuration needed to be...,Reviewed and centralized the required Pandas a...,Python/Pandas environment started successfully.
1,Section 2,Inherited core loading section required verifi...,Loaded orders.csv from the extracted DATA_DIR ...,orders_df loaded successfully with 5000 rows.
2,Section 3,Order status summary referenced incorrect colu...,Changed the status analysis to use the correct...,"Summary table, duplicate order count, missing-..."
3,Section 5,"Customer and seller filenames were incorrect, ...",Implemented a reusable FlipkartIssueDataLoader...,"Loaded datasets: ['orders', 'customers', 'sell..."
4,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
5,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
6,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
7,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
8,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
9,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...


## Section 9: Refund Delay and Refund-Priority Analysis

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `refunds.xlsx`
- `orders.csv`

**How to approach:**
- Parse refund dates.
- Calculate refund age.
- Detect pending/failed refunds.
- Detect refund SLA breaches and amount mismatches.

**Expected output format:**
- refund_summary table.
- refunds table with refund_priority_candidate and flags.

**Hint:**
- A pending refund needs age calculated against a reference date.

In [ ]:
# SECTION 9 — REFUND DELAY AND REFUND-PRIORITY ANALYSIS

refunds_analysis = refunds_clean.copy()

orders_for_refund = orders_clean[
    [
        "order_id",
        "order_amount",
        "delivery_delay_days",
        "delay_bucket"
    ]
].copy()


refunds_analysis["refund_initiated_date"] = pd.to_datetime(
    refunds_analysis["refund_initiated_date"],
    errors="coerce"
)

refunds_analysis["refund_completed_date"] = pd.to_datetime(
    refunds_analysis["refund_completed_date"],
    errors="coerce"
)


refunds_analysis["refund_status_clean"] = (
    refunds_analysis["refund_status"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(r"[\s-]+", "_", regex=True)
)


refunds_analysis["refund_amount_clean"] = pd.to_numeric(
    refunds_analysis["refund_amount"],
    errors="coerce"
)

reference_date = pd.Timestamp.today().normalize()

refunds_analysis["refund_reference_date"] = (
    refunds_analysis["refund_completed_date"]
    .fillna(reference_date)
)

refunds_analysis["refund_age_days"] = (
    refunds_analysis["refund_reference_date"]
    - refunds_analysis["refund_initiated_date"]
).dt.days

refunds_analysis["pending_refund_flag"] = (
    refunds_analysis["refund_status_clean"]
    .isin([
        "pending",
        "initiated",
        "in_progress"
    ])
)

refunds_analysis["failed_refund_flag"] = (
    refunds_analysis["refund_status_clean"]
    .isin([
        "failed",
        "rejected"
    ])
)

refunds_analysis["refund_sla_breach_flag"] = (
    refunds_analysis["refund_age_days"] > 7
)


refunds_analysis = refunds_analysis.merge(
    orders_for_refund,
    on="order_id",
    how="left"
)

refunds_analysis["amount_mismatch_flag"] = (
    refunds_analysis["refund_amount_clean"]
    > refunds_analysis["order_amount"]
)

refunds_analysis["delayed_order_refund_flag"] = (
    refunds_analysis["delivery_delay_days"] > 0
)

refunds_analysis["refund_priority_candidate"] = (
    refunds_analysis["pending_refund_flag"]
    | refunds_analysis["failed_refund_flag"]
    | refunds_analysis["refund_sla_breach_flag"]
    | refunds_analysis["amount_mismatch_flag"]
    | refunds_analysis["delayed_order_refund_flag"]
)

def calculate_refund_risk_score(row):
    score = 0

    if row["pending_refund_flag"]:
        score += 3

    if row["failed_refund_flag"]:
        score += 3

    if row["refund_sla_breach_flag"]:
        score += 3

    if row["amount_mismatch_flag"]:
        score += 2

    if row["delayed_order_refund_flag"]:
        score += 2

    return score


refunds_analysis["refund_risk_score"] = refunds_analysis.apply(
    calculate_refund_risk_score,
    axis=1
)

def classify_refund_risk(score):
    if score >= 6:
        return "High"
    elif score >= 3:
        return "Medium"
    else:
        return "Low"


refunds_analysis["refund_risk_level"] = (
    refunds_analysis["refund_risk_score"]
    .apply(classify_refund_risk)
)

refund_summary = pd.DataFrame({
    "Metric": [
        "Total Refund Records",
        "Pending / Initiated Refunds",
        "Failed / Rejected Refunds",
        "Refund SLA Breaches",
        "Amount Mismatches",
        "Delayed-Order Refunds",
        "Refund Priority Candidates",
        "High-Risk Refunds"
    ],
    "Count": [
        len(refunds_analysis),
        refunds_analysis["pending_refund_flag"].sum(),
        refunds_analysis["failed_refund_flag"].sum(),
        refunds_analysis["refund_sla_breach_flag"].sum(),
        refunds_analysis["amount_mismatch_flag"].sum(),
        refunds_analysis["delayed_order_refund_flag"].sum(),
        refunds_analysis["refund_priority_candidate"].sum(),
        (refunds_analysis["refund_risk_level"] == "High").sum()
    ]
})

print("REFUND SUMMARY")

display(refund_summary)

print("\nREFUND ANALYSIS PREVIEW")

refund_display_columns = [
    "refund_id",
    "order_id",
    "refund_status_clean",
    "refund_amount_clean",
    "refund_initiated_date",
    "refund_completed_date",
    "refund_age_days",
    "pending_refund_flag",
    "failed_refund_flag",
    "refund_sla_breach_flag",
    "amount_mismatch_flag",
    "delayed_order_refund_flag",
    "refund_priority_candidate",
    "refund_risk_score",
    "refund_risk_level"
]

display(
    refunds_analysis[
        [
            col for col in refund_display_columns
            if col in refunds_analysis.columns
        ]
    ].head(10)
)


add_debug_fix(
    section="Section 9",
    issue_found=(
        "Inherited refund logic used incorrect date column names and "
        "did not implement refund age, pending/failed flags, SLA breach, "
        "amount mismatch, or delayed-order linkage."
    ),
    fix_applied=(
        "Aligned the logic with the actual refund schema and implemented "
        "safe date parsing, reference-date refund age, refund SLA checks, "
        "amount mismatch checks, delayed-order linkage, and refund risk scoring."
    ),
    validation_output=(
        f"Refund analysis completed for {len(refunds_analysis)} records; "
        f"{refunds_analysis['refund_priority_candidate'].sum()} "
        "refund-priority candidates identified."
    )
)

display(get_debug_log())

REFUND SUMMARY


,Metric,Count
0,Total Refund Records,1310
1,Pending / Initiated Refunds,436
2,Failed / Rejected Refunds,221
3,Refund SLA Breaches,0
4,Amount Mismatches,251
5,Delayed-Order Refunds,0
6,Refund Priority Candidates,794
7,High-Risk Refunds,0



REFUND ANALYSIS PREVIEW


,refund_id,order_id,refund_status_clean,refund_amount_clean,refund_initiated_date,refund_completed_date,refund_age_days,pending_refund_flag,failed_refund_flag,refund_sla_breach_flag,amount_mismatch_flag,delayed_order_refund_flag,refund_priority_candidate,refund_risk_score,refund_risk_level
0,ref000001,ord0000498,rejected,44569.30,NaT,NaT,NaN,False,True,False,False,False,True,3,Medium
1,ref000002,ord0001547,initiated,40965.07,NaT,NaT,NaN,True,False,False,True,False,True,5,Medium
2,ref000003,ord0003579,processed,14072.25,NaT,NaT,NaN,False,False,False,False,False,False,0,Low
3,ref000004,ord0000632,pending,45492.77,NaT,NaT,NaN,True,False,False,False,False,True,3,Medium
4,ref000005,ord0000671,processed,42444.88,NaT,NaT,NaN,False,False,False,False,False,False,0,Low
5,ref000006,ord0000941,processed,27143.06,NaT,NaT,NaN,False,False,False,True,False,True,2,Low
6,ref000007,ord0003554,failed,9900.84,NaT,NaT,NaN,False,True,False,False,False,True,3,Medium
7,ref000008,ord0000024,pending,61122.63,NaT,NaT,NaN,True,False,False,False,False,True,3,Medium
8,ref000009,ord0001465,rejected,1720.73,NaT,NaT,NaN,False,True,False,False,False,True,3,Medium
9,ref000010,ord0000369,failed,17042.95,NaT,NaT,NaN,False,True,False,False,False,True,3,Medium


,section,issue_found,fix_applied,validation_output
0,Section 1,Imports and project configuration needed to be...,Reviewed and centralized the required Pandas a...,Python/Pandas environment started successfully.
1,Section 2,Inherited core loading section required verifi...,Loaded orders.csv from the extracted DATA_DIR ...,orders_df loaded successfully with 5000 rows.
2,Section 3,Order status summary referenced incorrect colu...,Changed the status analysis to use the correct...,"Summary table, duplicate order count, missing-..."
3,Section 5,"Customer and seller filenames were incorrect, ...",Implemented a reusable FlipkartIssueDataLoader...,"Loaded datasets: ['orders', 'customers', 'sell..."
4,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
5,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
6,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
7,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
8,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
9,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...


## Section 10: Customer Complaint Linking

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `customer_complaints.json`
- `orders.csv`
- `customers.csv`

**How to approach:**
- Aggregate complaints by order_id.
- Count negative or angry complaints.
- Detect repeated complaints.
- Create complaint type summary.

**Expected output format:**
- complaints_order table.
- complaint_type_summary table.

**Hint:**
- Aggregate before merging into the order-level feature table.

In [ ]:
# SECTION 10 — CUSTOMER COMPLAINT LINKING

complaints_df = pd.DataFrame(complaints_data).copy()

complaints_df["order_id"] = (
    complaints_df["order_id"]
    .astype("string")
    .str.strip()
)

complaints_df["customer_id"] = (
    complaints_df["customer_id"]
    .astype("string")
    .str.strip()
)

complaints_df["complaint_type"] = (
    complaints_df["complaint_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

complaints_df["sentiment_tag"] = (
    complaints_df["sentiment_tag"]
    .astype("string")
    .str.strip()
    .str.lower()
)

negative_sentiments = {
    "negative",
    "angry",
    "frustrated"
}

angry_sentiments = {
    "angry"
}

complaints_df["negative_complaint_flag"] = (
    complaints_df["sentiment_tag"]
    .isin(negative_sentiments)
)

complaints_df["angry_complaint_flag"] = (
    complaints_df["sentiment_tag"]
    .isin(angry_sentiments)
)

complaints_order = (
    complaints_df
    .groupby("order_id", dropna=False)
    .agg(
        complaint_count=("complaint_id", "count"),
        negative_complaint_count=(
            "negative_complaint_flag",
            "sum"
        ),
        angry_complaint_count=(
            "angry_complaint_flag",
            "sum"
        ),
        unique_complaint_types=(
            "complaint_type",
            "nunique"
        ),
        customer_id_from_complaint=(
            "customer_id",
            "first"
        )
    )
    .reset_index()
)

complaints_order["repeated_complaint_flag"] = (
    complaints_order["complaint_count"] > 1
)

complaints_order["complaint_risk_flag"] = (
    (complaints_order["negative_complaint_count"] > 0)
    |
    (complaints_order["repeated_complaint_flag"])
)

complaint_type_summary = (
    complaints_df
    .groupby("complaint_type", dropna=False)
    .agg(
        complaint_count=("complaint_id", "count"),
        unique_orders=("order_id", "nunique"),
        negative_complaints=(
            "negative_complaint_flag",
            "sum"
        ),
        angry_complaints=(
            "angry_complaint_flag",
            "sum"
        )
    )
    .reset_index()
    .sort_values(
        by="complaint_count",
        ascending=False
    )
)

print("COMPLAINTS BY ORDER")

display(complaints_order.head(10))

print("\nCOMPLAINT TYPE SUMMARY")

display(complaint_type_summary)


add_debug_fix(
    section="Section 10",
    issue_found=(
        "Inherited complaint logic used a non-existent 'order' column "
        "and did not aggregate complaints or identify repeated/negative signals."
    ),
    fix_applied=(
        "Changed grouping to order_id and created order-level complaint "
        "features, repeated complaint flags, negative/angry sentiment flags, "
        "and complaint type summaries before downstream merging."
    ),
    validation_output=(
        f"Complaint aggregation completed for {complaints_order['order_id'].nunique()} "
        f"orders and {len(complaint_type_summary)} complaint types."
    )
)

display(get_debug_log())

COMPLAINTS BY ORDER


,order_id,complaint_count,negative_complaint_count,angry_complaint_count,unique_complaint_types,customer_id_from_complaint,repeated_complaint_flag,complaint_risk_flag
0,ORD0000001,2,1,0,2,CUST000771,True,True
1,ORD0000006,1,0,0,1,CUST001093,False,False
2,ORD0000007,2,1,1,2,CUST000829,True,True
3,ORD0000018,1,1,0,1,CUST000533,False,True
4,ORD0000024,1,1,1,1,CUST000826,False,True
5,ORD0000025,1,1,0,1,CUST000492,False,True
6,ORD0000027,1,1,0,1,CUST000556,False,True
7,ORD0000028,1,0,0,1,CUST000321,False,False
8,ORD0000039,1,1,1,1,CUST000849,False,True
9,ORD0000044,1,1,0,1,CUST001118,False,True



COMPLAINT TYPE SUMMARY


,complaint_type,complaint_count,unique_orders,negative_complaints,angry_complaints
5,refund delay,196,194,143,73
0,cancellation issue,194,190,128,52
1,damaged product,194,186,114,38
7,wrong item,193,193,118,36
2,late delivery,187,185,123,47
4,payment deducted twice,183,178,112,37
6,seller issue,177,173,107,32
3,missing item,176,175,114,40


,section,issue_found,fix_applied,validation_output
0,Section 1,Imports and project configuration needed to be...,Reviewed and centralized the required Pandas a...,Python/Pandas environment started successfully.
1,Section 2,Inherited core loading section required verifi...,Loaded orders.csv from the extracted DATA_DIR ...,orders_df loaded successfully with 5000 rows.
2,Section 3,Order status summary referenced incorrect colu...,Changed the status analysis to use the correct...,"Summary table, duplicate order count, missing-..."
3,Section 5,"Customer and seller filenames were incorrect, ...",Implemented a reusable FlipkartIssueDataLoader...,"Loaded datasets: ['orders', 'customers', 'sell..."
4,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
5,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
6,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
7,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
8,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
9,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...


## Section 11: Support Ticket Mapping

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `support_tickets.csv`
- `orders.csv`

**How to approach:**
- Standardize ticket status.
- Flag escalations.
- Create SLA breach logic.
- Aggregate ticket signals by order_id.

**Expected output format:**
- tickets_order table.
- ticket_team_summary table.

**Hint:**
- Use `resolution_time_hours`, not `resolution_hours`.

In [ ]:
# SECTION 11 — SUPPORT TICKET MAPPING

tickets_df = support_tickets_df.copy()

if "ticket_status" in tickets_df.columns:

    tickets_df["ticket_status_clean"] = (
        tickets_df["ticket_status"]
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"[\s-]+", "_", regex=True)
    )

else:

    tickets_df["ticket_status_clean"] = "unknown"

tickets_df["escalation_flag_clean"] = (
    tickets_df["escalation_flag"]
    .astype("string")
    .str.strip()
    .str.lower()
)

tickets_df["escalated_ticket_flag"] = (
    tickets_df["escalation_flag_clean"]
    .isin([
        "yes",
        "true",
        "1"
    ])
)

tickets_df["resolution_time_hours_clean"] = (
    pd.to_numeric(
        tickets_df["resolution_time_hours"],
        errors="coerce"
    )
)

tickets_df["ticket_sla_breach_flag"] = (
    tickets_df["resolution_time_hours_clean"] > 48
)

tickets_df["missing_resolution_time_flag"] = (
    tickets_df["escalated_ticket_flag"]
    &
    tickets_df["resolution_time_hours_clean"].isna()
)

tickets_df["ticket_sla_breach_flag"] = (
    tickets_df["ticket_sla_breach_flag"]
    |
    tickets_df["missing_resolution_time_flag"]
)

tickets_df["ticket_risk_flag"] = (
    tickets_df["escalated_ticket_flag"]
    |
    tickets_df["ticket_sla_breach_flag"]
)

tickets_order = (
    tickets_df
    .groupby(
        "order_id",
        dropna=False
    )
    .agg(
        ticket_count=(
            "ticket_id",
            "count"
        ),
        escalated_ticket_count=(
            "escalated_ticket_flag",
            "sum"
        ),
        ticket_sla_breach_count=(
            "ticket_sla_breach_flag",
            "sum"
        ),
        missing_resolution_time_count=(
            "missing_resolution_time_flag",
            "sum"
        ),
        ticket_risk_count=(
            "ticket_risk_flag",
            "sum"
        ),
        average_resolution_time_hours=(
            "resolution_time_hours_clean",
            "mean"
        ),
        max_resolution_time_hours=(
            "resolution_time_hours_clean",
            "max"
        )
    )
    .reset_index()
)

tickets_order[
    "average_resolution_time_hours"
] = (
    tickets_order[
        "average_resolution_time_hours"
    ]
    .round(2)
)

tickets_order[
    "max_resolution_time_hours"
] = (
    tickets_order[
        "max_resolution_time_hours"
    ]
    .round(2)
)

tickets_order[
    "repeated_support_issue_flag"
] = (
    tickets_order[
        "ticket_count"
    ] > 1
)

if "assigned_team" in tickets_df.columns:

    ticket_team_summary = (
        tickets_df
        .groupby(
            "assigned_team",
            dropna=False
        )
        .agg(
            ticket_count=(
                "ticket_id",
                "count"
            ),
            unique_orders=(
                "order_id",
                "nunique"
            ),
            escalated_tickets=(
                "escalated_ticket_flag",
                "sum"
            ),
            sla_breached_tickets=(
                "ticket_sla_breach_flag",
                "sum"
            ),
            missing_resolution_time=(
                "missing_resolution_time_flag",
                "sum"
            ),
            average_resolution_time_hours=(
                "resolution_time_hours_clean",
                "mean"
            )
        )
        .reset_index()
    )

    ticket_team_summary[
        "average_resolution_time_hours"
    ] = (
        ticket_team_summary[
            "average_resolution_time_hours"
        ]
        .round(2)
    )

    ticket_team_summary = (
        ticket_team_summary
        .sort_values(
            by=[
                "escalated_tickets",
                "sla_breached_tickets"
            ],
            ascending=False
        )
    )

else:

    ticket_team_summary = pd.DataFrame({
        "assigned_team": [],
        "ticket_count": [],
        "unique_orders": [],
        "escalated_tickets": [],
        "sla_breached_tickets": [],
        "missing_resolution_time": [],
        "average_resolution_time_hours": []
    })


print("TICKETS ORDER SUMMARY")

display(
    tickets_order.head(10)
)

print("\nTICKET TEAM SUMMARY")

display(
    ticket_team_summary
)

print("\nSUPPORT TICKET VALIDATION")

print(
    "Total tickets:",
    len(tickets_df)
)

print(
    "Unique ticket IDs:",
    tickets_df["ticket_id"].nunique()
)

print(
    "Order-level ticket groups:",
    tickets_order["order_id"].nunique()
)

print(
    "Escalated tickets:",
    int(
        tickets_df[
            "escalated_ticket_flag"
        ].sum()
    )
)

print(
    "SLA-breached tickets:",
    int(
        tickets_df[
            "ticket_sla_breach_flag"
        ].sum()
    )
)

print(
    "Missing resolution time on escalated tickets:",
    int(
        tickets_df[
            "missing_resolution_time_flag"
        ].sum()
    )
)


add_debug_fix(
    section="Section 11",
    issue_found=(
        "Support ticket mapping was incomplete and used the wrong "
        "resolution-time field assumption."
    ),
    fix_applied=(
        "Used resolution_time_hours, standardized ticket status and "
        "escalation flags, added SLA-breach logic, and aggregated "
        "ticket signals by order_id and assigned team."
    ),
    validation_output=(
        f"Processed {len(tickets_df)} tickets and created "
        f"{len(tickets_order)} order-level ticket groups."
    )
)

display(get_debug_log())

TICKETS ORDER SUMMARY


,order_id,ticket_count,escalated_ticket_count,ticket_sla_breach_count,missing_resolution_time_count,ticket_risk_count,average_resolution_time_hours,max_resolution_time_hours,repeated_support_issue_flag
0,ORD0000005,3,1,2,0,3,76.33,114.0,True
1,ORD0000012,1,1,0,0,1,31.00,31.0,False
2,ORD0000014,2,0,2,0,2,131.50,168.0,True
3,ORD0000016,1,0,1,0,1,60.00,60.0,False
4,ORD0000021,1,0,1,0,1,50.00,50.0,False
5,ORD0000023,1,0,0,0,0,48.00,48.0,False
6,ORD0000024,1,0,1,0,1,93.00,93.0,False
7,ORD0000026,2,0,1,0,1,95.00,168.0,True
8,ORD0000032,2,0,0,0,0,43.00,43.0,True
9,ORD0000036,1,0,1,0,1,155.00,155.0,False



TICKET TEAM SUMMARY


,assigned_team,ticket_count,unique_orders,escalated_tickets,sla_breached_tickets,missing_resolution_time,average_resolution_time_hours
3,Refund Ops,264,260,57,193,9,95.91
2,Logistics Ops,253,247,54,175,5,94.37
0,City Ops,260,253,52,181,6,90.72
5,Support Ops,234,227,52,164,4,89.82
4,Seller Ops,267,260,47,185,7,92.56
1,Escalation Desk,222,215,42,147,4,87.38



SUPPORT TICKET VALIDATION
Total tickets: 1500
Unique ticket IDs: 1500
Order-level ticket groups: 1287
Escalated tickets: 304
SLA-breached tickets: 1045
Missing resolution time on escalated tickets: 35


,section,issue_found,fix_applied,validation_output
0,Section 1,Imports and project configuration needed to be...,Reviewed and centralized the required Pandas a...,Python/Pandas environment started successfully.
1,Section 2,Inherited core loading section required verifi...,Loaded orders.csv from the extracted DATA_DIR ...,orders_df loaded successfully with 5000 rows.
2,Section 3,Order status summary referenced incorrect colu...,Changed the status analysis to use the correct...,"Summary table, duplicate order count, missing-..."
3,Section 5,"Customer and seller filenames were incorrect, ...",Implemented a reusable FlipkartIssueDataLoader...,"Loaded datasets: ['orders', 'customers', 'sell..."
4,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
5,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
6,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
7,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
8,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
9,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...


## Section 12: Duplicate and Repeat Issue Detection

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `orders.csv`
- `customer_complaints.json`
- `support_tickets.csv`

**How to approach:**
- Find duplicate order IDs.
- Find duplicate-looking order patterns.
- Find customers with repeated complaints.
- Summarize issue counts.

**Expected output format:**
- duplicate_summary table.
- Optional preview of repeated complaint customers.

**Hint:**
- Duplicate-looking records may share customer, seller, date, and amount even if row IDs differ.

In [ ]:
# SECTION 12 — DUPLICATE AND REPEAT ISSUE DETECTION

duplicate_check_orders = orders_clean.copy()

exact_duplicate_rows = duplicate_check_orders.duplicated(
    keep=False
).sum()

duplicate_order_id_mask = duplicate_check_orders[
    "order_id"
].duplicated(keep=False)

duplicate_order_id_rows = duplicate_order_id_mask.sum()

duplicate_order_ids = duplicate_check_orders.loc[
    duplicate_order_id_mask,
    "order_id"
].nunique()

duplicate_pattern_columns = [
    "customer_id",
    "seller_id",
    "order_date",
    "order_amount"
]

available_pattern_columns = [
    col
    for col in duplicate_pattern_columns
    if col in duplicate_check_orders.columns
]

if len(available_pattern_columns) == len(duplicate_pattern_columns):

    duplicate_check_orders["duplicate_pattern_count"] = (
        duplicate_check_orders
        .groupby(available_pattern_columns, dropna=False)["order_id"]
        .transform("count")
    )

    duplicate_looking_mask = (
        duplicate_check_orders["duplicate_pattern_count"] > 1
    )

else:

    duplicate_check_orders["duplicate_pattern_count"] = 1
    duplicate_looking_mask = pd.Series(
        False,
        index=duplicate_check_orders.index
    )

duplicate_looking_rows = duplicate_looking_mask.sum()

duplicate_looking_orders = duplicate_check_orders.loc[
    duplicate_looking_mask,
    [
        "order_id",
        "customer_id",
        "seller_id",
        "order_date",
        "order_amount",
        "duplicate_pattern_count"
    ]
].sort_values(
    by="duplicate_pattern_count",
    ascending=False
)

complaints_repeat_by_customer = (
    complaints_df
    .groupby("customer_id", dropna=False)
    .agg(
        complaint_count=("complaint_id", "count"),
        unique_orders=("order_id", "nunique"),
        angry_complaints=(
            "angry_complaint_flag",
            "sum"
        ),
        negative_complaints=(
            "negative_complaint_flag",
            "sum"
        )
    )
    .reset_index()
)

complaints_repeat_by_customer[
    "repeated_complaint_customer_flag"
] = (
    complaints_repeat_by_customer["complaint_count"] > 1
)

repeated_complaint_customers = (
    complaints_repeat_by_customer[
        complaints_repeat_by_customer[
            "repeated_complaint_customer_flag"
        ]
    ]
    .sort_values(
        by="complaint_count",
        ascending=False
    )
)

support_repeat_by_order = (
    support_tickets_clean
    .groupby("order_id", dropna=False)
    .agg(
        ticket_count=("ticket_id", "count"),
        escalated_ticket_count=(
            "escalation_flag",
            lambda x: (
                x.astype("string")
                .str.strip()
                .str.lower()
                .isin(["yes", "true", "1"])
                .sum()
            )
        )
    )
    .reset_index()
)

support_repeat_by_order[
    "repeated_support_issue_flag"
] = (
    support_repeat_by_order["ticket_count"] > 1
)

duplicate_summary = pd.DataFrame({
    "issue_type": [
        "Exact duplicate rows",
        "Duplicate order IDs",
        "Duplicate order ID rows",
        "Duplicate-looking order rows",
        "Customers with repeated complaints",
        "Orders with repeated support issues"
    ],
    "issue_count": [
        exact_duplicate_rows,
        duplicate_order_ids,
        duplicate_order_id_rows,
        duplicate_looking_rows,
        len(repeated_complaint_customers),
        support_repeat_by_order[
            "repeated_support_issue_flag"
        ].sum()
    ]
})


print("DUPLICATE AND REPEAT ISSUE SUMMARY")

display(duplicate_summary)


print("\nDUPLICATE-LOOKING ORDER PREVIEW")

if len(duplicate_looking_orders) > 0:
    display(
        duplicate_looking_orders.head(10)
    )
else:
    print("No duplicate-looking order patterns detected.")


print("\nCUSTOMERS WITH REPEATED COMPLAINTS")

if len(repeated_complaint_customers) > 0:
    display(
        repeated_complaint_customers.head(10)
    )
else:
    print("No customers with repeated complaints detected.")


print("\nORDERS WITH REPEATED SUPPORT ISSUES")

repeated_support_orders = support_repeat_by_order[
    support_repeat_by_order["repeated_support_issue_flag"]
].sort_values(
    by="ticket_count",
    ascending=False
)

if len(repeated_support_orders) > 0:
    display(
        repeated_support_orders.head(10)
    )
else:
    print("No orders with repeated support issues detected.")



add_debug_fix(
    section="Section 12",
    issue_found=(
        "Inherited logic checked only exact duplicate rows and did not "
        "identify duplicate order IDs, duplicate-looking records, repeated "
        "customer complaints, or repeated support issues."
    ),
    fix_applied=(
        "Added duplicate order ID checks, duplicate-looking order pattern "
        "detection using customer/seller/date/amount, repeated complaint "
        "customer analysis, and repeated support issue analysis."
    ),
    validation_output=(
        f"Duplicate/repeat analysis completed with "
        f"{len(duplicate_summary)} issue checks."
    )
)

display(get_debug_log())


DUPLICATE AND REPEAT ISSUE SUMMARY


,issue_type,issue_count
0,Exact duplicate rows,0
1,Duplicate order IDs,16
2,Duplicate order ID rows,32
3,Duplicate-looking order rows,0
4,Customers with repeated complaints,412
5,Orders with repeated support issues,197



DUPLICATE-LOOKING ORDER PREVIEW
No duplicate-looking order patterns detected.

CUSTOMERS WITH REPEATED COMPLAINTS


,customer_id,complaint_count,unique_orders,angry_complaints,negative_complaints,repeated_complaint_customer_flag
235,CUST000364,7,5,0,6,True
651,CUST000973,6,6,0,4,True
561,CUST000829,6,4,2,2,True
443,CUST000664,6,4,1,2,True
43,CUST000064,5,2,1,3,True
747,CUST001109,5,3,2,2,True
56,CUST000079,5,4,1,2,True
737,CUST001098,5,4,0,2,True
547,CUST000811,5,4,2,3,True
227,CUST000356,5,3,2,4,True



ORDERS WITH REPEATED SUPPORT ISSUES


,order_id,ticket_count,escalated_ticket_count,repeated_support_issue_flag
856,ord0003389,4,2,True
15,ord0000054,3,1,True
143,ord0000558,3,1,True
63,ord0000246,3,0,True
0,ord0000005,3,1,True
329,ord0001319,3,1,True
268,ord0001048,3,0,True
986,ord0003923,3,0,True
1059,ord0004190,3,0,True
549,ord0002132,3,0,True


,section,issue_found,fix_applied,validation_output
0,Section 1,Imports and project configuration needed to be...,Reviewed and centralized the required Pandas a...,Python/Pandas environment started successfully.
1,Section 2,Inherited core loading section required verifi...,Loaded orders.csv from the extracted DATA_DIR ...,orders_df loaded successfully with 5000 rows.
2,Section 3,Order status summary referenced incorrect colu...,Changed the status analysis to use the correct...,"Summary table, duplicate order count, missing-..."
3,Section 5,"Customer and seller filenames were incorrect, ...",Implemented a reusable FlipkartIssueDataLoader...,"Loaded datasets: ['orders', 'customers', 'sell..."
4,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
5,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
6,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
7,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
8,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
9,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...


## Section 13: Final Feature Table / Master Merge

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `orders.csv`
- `customers.csv`
- `sellers.csv`
- `refunds.xlsx`
- `customer_complaints.json`
- `support_tickets.csv`

**How to approach:**
- Aggregate many-to-one tables first.
- Merge customer and seller master data.
- Merge aggregated refund/complaint/ticket features.
- Validate row count before and after merge.

**Expected output format:**
- final_feature_table DataFrame.
- merge_validation table.

**Hint:**
- Raw refund, complaint, and ticket merges can multiply rows if not aggregated first.

In [ ]:
# SECTION 13 — FINAL FEATURE TABLE / MASTER MERGE

orders_before_merge = len(orders_clean)
unique_orders_before_merge = orders_clean["order_id"].nunique()

order_feature_columns = [
    "order_id",
    "customer_id",
    "seller_id",
    "delivery_city",
    "order_amount",
    "order_status_clean",
    "delivery_delay_days",
    "delay_bucket"
]

available_order_columns = [
    col for col in order_feature_columns
    if col in orders_clean.columns
]

orders_base = (
    orders_clean[available_order_columns]
    .groupby("order_id", as_index=False)
    .first()
)

refunds_order = (
    refunds_analysis
    .groupby("order_id", dropna=False)
    .agg(
        refund_count=("refund_id", "count"),
        pending_refund_count=(
            "pending_refund_flag",
            "sum"
        ),
        failed_refund_count=(
            "failed_refund_flag",
            "sum"
        ),
        refund_sla_breach_count=(
            "refund_sla_breach_flag",
            "sum"
        ),
        refund_amount_mismatch_count=(
            "amount_mismatch_flag",
            "sum"
        ),
        delayed_order_refund_count=(
            "delayed_order_refund_flag",
            "sum"
        ),
        refund_priority_candidate=(
            "refund_priority_candidate",
            "sum"
        ),
        max_refund_risk_score=(
            "refund_risk_score",
            "max"
        )
    )
    .reset_index()
)

complaints_order_features = complaints_order[
    [
        "order_id",
        "complaint_count",
        "negative_complaint_count",
        "angry_complaint_count",
        "repeated_complaint_flag",
        "complaint_risk_flag"
    ]
].copy()

support_tickets_clean["escalation_flag_clean"] = (
    support_tickets_clean["escalation_flag"]
    .astype("string")
    .str.strip()
    .str.lower()
)

support_tickets_clean["escalated_ticket_flag"] = (
    support_tickets_clean["escalation_flag_clean"]
    .isin(["yes", "true", "1"])
)

if "resolution_time_hours" in support_tickets_clean.columns:

    support_tickets_clean["ticket_sla_breach_flag"] = (
        pd.to_numeric(
            support_tickets_clean["resolution_time_hours"],
            errors="coerce"
        ) > 48
    )

else:

    support_tickets_clean["ticket_sla_breach_flag"] = False


tickets_order_features = (
    support_tickets_clean
    .groupby("order_id", dropna=False)
    .agg(
        ticket_count=("ticket_id", "count"),
        escalated_ticket_count=(
            "escalated_ticket_flag",
            "sum"
        ),
        ticket_sla_breach_count=(
            "ticket_sla_breach_flag",
            "sum"
        ),
        repeated_support_issue_flag=(
            "ticket_id",
            lambda x: len(x) > 1
        )
    )
    .reset_index()
)

customer_columns = [
    "customer_id",
    "segment",
    "account_age_days",
    "total_orders",
    "payment_preference"
]

available_customer_columns = [
    col for col in customer_columns
    if col in customers_clean.columns
]

customers_master = (
    customers_clean[available_customer_columns]
    .drop_duplicates("customer_id")
    .copy()
)

seller_columns = [
    "seller_id",
    "seller_category",
    "city",
    "rating",
    "monthly_order_volume"
]

available_seller_columns = [
    col
    for col in seller_columns
    if col in sellers_clean.columns
]

sellers_master = (
    sellers_clean[available_seller_columns]
    .drop_duplicates("seller_id")
    .copy()
)

base_row_count = len(orders_base)
base_unique_order_count = orders_base["order_id"].nunique()

final_feature_table = orders_base.merge(
    customers_master,
    on="customer_id",
    how="left",
    suffixes=("", "_customer"),
    validate="many_to_one"
)

rows_after_customer_merge = len(final_feature_table)

final_feature_table = final_feature_table.merge(
    sellers_master,
    on="seller_id",
    how="left",
    suffixes=("", "_seller"),
    validate="many_to_one"
)

rows_after_seller_merge = len(final_feature_table)

final_feature_table = final_feature_table.merge(
    refunds_order,
    on="order_id",
    how="left",
    validate="one_to_one"
)

rows_after_refund_merge = len(final_feature_table)


final_feature_table = final_feature_table.merge(
    complaints_order_features,
    on="order_id",
    how="left",
    validate="one_to_one"
)

rows_after_complaint_merge = len(final_feature_table)

final_feature_table = final_feature_table.merge(
    tickets_order_features,
    on="order_id",
    how="left",
    validate="one_to_one"
)

rows_after_ticket_merge = len(final_feature_table)

count_columns = [
    "refund_count",
    "pending_refund_count",
    "failed_refund_count",
    "refund_sla_breach_count",
    "refund_amount_mismatch_count",
    "delayed_order_refund_count",
    "refund_priority_candidate",
    "complaint_count",
    "negative_complaint_count",
    "angry_complaint_count",
    "ticket_count",
    "escalated_ticket_count",
    "ticket_sla_breach_count"
]

for column in count_columns:
    if column in final_feature_table.columns:
        final_feature_table[column] = (
            pd.to_numeric(
                final_feature_table[column],
                errors="coerce"
            )
            .fillna(0)
            .astype(int)
        )


flag_columns = [
    "repeated_complaint_flag",
    "complaint_risk_flag",
    "repeated_support_issue_flag"
]

for column in flag_columns:
    if column in final_feature_table.columns:
        final_feature_table[column] = (
            final_feature_table[column]
            .fillna(False)
            .astype(bool)
        )

final_row_count = len(final_feature_table)
final_unique_order_count = final_feature_table["order_id"].nunique()

duplicate_final_order_ids = (
    final_feature_table["order_id"]
    .duplicated()
    .sum()
)

merge_validation = pd.DataFrame([
    {
        "check_name": "Base order rows",
        "before_rows": orders_before_merge,
        "after_rows": base_row_count,
        "status": (
            "PASS"
            if base_row_count == base_unique_order_count
            else "REVIEW"
        ),
        "notes": (
            f"{unique_orders_before_merge} unique order IDs detected."
        )
    },
    {
        "check_name": "Customer master merge",
        "before_rows": base_row_count,
        "after_rows": rows_after_customer_merge,
        "status": (
            "PASS"
            if rows_after_customer_merge == base_row_count
            else "FAIL"
        ),
        "notes": "Customer master merged with many-to-one validation."
    },
    {
        "check_name": "Seller master merge",
        "before_rows": rows_after_customer_merge,
        "after_rows": rows_after_seller_merge,
        "status": (
            "PASS"
            if rows_after_seller_merge == rows_after_customer_merge
            else "FAIL"
        ),
        "notes": "Seller master merged with many-to-one validation."
    },
    {
        "check_name": "Aggregated refund merge",
        "before_rows": rows_after_seller_merge,
        "after_rows": rows_after_refund_merge,
        "status": (
            "PASS"
            if rows_after_refund_merge == rows_after_seller_merge
            else "FAIL"
        ),
        "notes": "Refunds were aggregated by order_id before merging."
    },
    {
        "check_name": "Aggregated complaint merge",
        "before_rows": rows_after_refund_merge,
        "after_rows": rows_after_complaint_merge,
        "status": (
            "PASS"
            if rows_after_complaint_merge == rows_after_refund_merge
            else "FAIL"
        ),
        "notes": "Complaints were aggregated by order_id before merging."
    },
    {
        "check_name": "Aggregated ticket merge",
        "before_rows": rows_after_complaint_merge,
        "after_rows": rows_after_ticket_merge,
        "status": (
            "PASS"
            if rows_after_ticket_merge == rows_after_complaint_merge
            else "FAIL"
        ),
        "notes": "Tickets were aggregated by order_id before merging."
    },
    {
        "check_name": "Final unique order IDs",
        "before_rows": base_unique_order_count,
        "after_rows": final_unique_order_count,
        "status": (
            "PASS"
            if final_unique_order_count == base_unique_order_count
            else "FAIL"
        ),
        "notes": "Final feature table should contain one row per order."
    },
    {
        "check_name": "Final duplicate order IDs",
        "before_rows": 0,
        "after_rows": duplicate_final_order_ids,
        "status": (
            "PASS"
            if duplicate_final_order_ids == 0
            else "FAIL"
        ),
        "notes": "Checks for accidental row multiplication after merges."
    }
])

print("MERGE VALIDATION")

display(merge_validation)

print("\nFINAL FEATURE TABLE")

print("Rows:", len(final_feature_table))
print("Columns:", len(final_feature_table.columns))
print("Unique orders:", final_feature_table["order_id"].nunique())

display(final_feature_table.head(10))


print("\nFinal duplicate order IDs:", duplicate_final_order_ids)

if duplicate_final_order_ids == 0:
    print("Row-level merge validation: PASS")
else:
    print("Row-level merge validation: FAIL — investigate row multiplication.")


add_debug_fix(
    section="Section 13",
    issue_found=(
        "Inherited code merged raw refund, complaint, and ticket records "
        "directly into orders, which could multiply order rows."
    ),
    fix_applied=(
        "Aggregated refunds, complaints, and support tickets to order level "
        "before merging. Customer and seller master tables were merged with "
        "many-to-one validation and row counts were checked after every merge."
    ),
    validation_output=(
        f"Final feature table contains {len(final_feature_table)} rows, "
        f"{final_feature_table['order_id'].nunique()} unique orders, and "
        f"{duplicate_final_order_ids} duplicate final order IDs."
    )
)

display(get_debug_log())

MERGE VALIDATION


/tmp/ipykernel_2252/991475228.py:251: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
/tmp/ipykernel_2252/991475228.py:251: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)
/tmp/ipykernel_2252/991475228.py:251: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


,check_name,before_rows,after_rows,status,notes
0,Base order rows,5000,4984,PASS,4984 unique order IDs detected.
1,Customer master merge,4984,4984,PASS,Customer master merged with many-to-one valida...
2,Seller master merge,4984,4984,PASS,Seller master merged with many-to-one validation.
3,Aggregated refund merge,4984,4984,PASS,Refunds were aggregated by order_id before mer...
4,Aggregated complaint merge,4984,4984,PASS,Complaints were aggregated by order_id before ...
5,Aggregated ticket merge,4984,4984,PASS,Tickets were aggregated by order_id before mer...
6,Final unique order IDs,4984,4984,PASS,Final feature table should contain one row per...
7,Final duplicate order IDs,0,0,PASS,Checks for accidental row multiplication after...



FINAL FEATURE TABLE
Rows: 4984
Columns: 29
Unique orders: 4984


,order_id,customer_id,seller_id,delivery_city,order_amount,order_status_clean,delivery_delay_days,delay_bucket,account_age_days,total_orders,...,max_refund_risk_score,complaint_count,negative_complaint_count,angry_complaint_count,repeated_complaint_flag,complaint_risk_flag,ticket_count,escalated_ticket_count,ticket_sla_breach_count,repeated_support_issue_flag
0,ord0000001,cust000771,sell0475,delhi,57056.16,delivered,NaN,Delivery Date Missing or Invalid,584.0,56.0,...,NaN,0,0,0,False,False,0,0,0,False
1,ord0000002,cust000122,sell0035,gurugram,25038.34,delivered,NaN,Delivery Date Missing or Invalid,940.0,71.0,...,3.0,0,0,0,False,False,0,0,0,False
2,ord0000003,cust000784,sell0785,lucknow,29439.12,processing,NaN,Delivery Date Missing or Invalid,1504.0,39.0,...,NaN,0,0,0,False,False,0,0,0,False
3,ord0000004,cust000288,sell0211,indore,60697.51,delivered,NaN,Delivery Date Missing or Invalid,862.0,49.0,...,NaN,0,0,0,False,False,0,0,0,False
4,ord0000005,cust001168,sell0981,nagpur,56251.50,cancelled,NaN,Delivery Date Missing or Invalid,1131.0,46.0,...,3.0,0,0,0,False,False,3,1,2,True
5,ord0000006,cust001093,sell0225,pune,34649.02,delivered,NaN,Delivery Date Missing or Invalid,1087.0,84.0,...,3.0,0,0,0,False,False,0,0,0,False
6,ord0000007,cust000829,sell0157,delhi,24879.38,delivered,NaN,Delivery Date Missing or Invalid,608.0,26.0,...,NaN,0,0,0,False,False,0,0,0,False
7,ord0000008,cust000980,sell0319,chennai,20175.42,cancelled,NaN,Delivery Date Missing or Invalid,1239.0,57.0,...,3.0,0,0,0,False,False,0,0,0,False
8,ord0000009,cust001004,sell0719,noida,31226.45,refund pending,NaN,Delivery Date Missing or Invalid,1178.0,86.0,...,NaN,0,0,0,False,False,0,0,0,False
9,ord0000010,custx960572,sell0202,chennai,58844.06,cancelled,NaN,Delivery Date Missing or Invalid,NaN,NaN,...,2.0,0,0,0,False,False,0,0,0,False



Final duplicate order IDs: 0
Row-level merge validation: PASS


,section,issue_found,fix_applied,validation_output
0,Section 1,Imports and project configuration needed to be...,Reviewed and centralized the required Pandas a...,Python/Pandas environment started successfully.
1,Section 2,Inherited core loading section required verifi...,Loaded orders.csv from the extracted DATA_DIR ...,orders_df loaded successfully with 5000 rows.
2,Section 3,Order status summary referenced incorrect colu...,Changed the status analysis to use the correct...,"Summary table, duplicate order count, missing-..."
3,Section 5,"Customer and seller filenames were incorrect, ...",Implemented a reusable FlipkartIssueDataLoader...,"Loaded datasets: ['orders', 'customers', 'sell..."
4,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
5,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
6,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
7,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
8,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
9,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...


## Section 14: Priority Classification

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `final_feature_table`

**How to approach:**
- Create a scoring function.
- Combine delay, refund, complaint, ticket, and customer segment signals.
- Return priority level and reasons.

**Expected output format:**
- priority_summary table.
- final_feature_table with issue_score, issue_priority, priority_reasons.

**Hint:**
- Priority should not be based on delay alone.

In [ ]:
# SECTION 14 — PRIORITY CLASSIFICATION

priority_df = final_feature_table.copy()


def safe_number(row, column, default=0):
    value = row.get(column, default)

    if pd.isna(value):
        return default

    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def safe_flag(row, column):
    value = row.get(column, False)

    if pd.isna(value):
        return False

    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    return str(value).strip().lower() in {
        "true",
        "yes",
        "1"
    }

def classify_issue_priority(row):
    """
    Calculate issue score using multiple business signals.

    Signals:
    - Delivery delay
    - Refund risk
    - Negative/angry complaints
    - Escalated/SLA-risk tickets
    - Repeated complaints/support issues
    - Customer segment / customer impact
    """

    score = 0
    reasons = []



    delay_days = safe_number(
        row,
        "delivery_delay_days"
    )

    delay_bucket = str(
        row.get("delay_bucket", "")
    ).strip().lower()

    if delay_bucket == "severe delay":
        score += 5
        reasons.append(
            "Severe delivery delay"
        )

    elif delay_bucket == "moderate delay":
        score += 3
        reasons.append(
            "Moderate delivery delay"
        )

    elif delay_bucket == "minor delay":
        score += 1
        reasons.append(
            "Minor delivery delay"
        )

    elif delay_days > 0:
        score += 1
        reasons.append(
            "Delivery promise breached"
        )



    pending_refunds = safe_number(
        row,
        "pending_refund_count"
    )

    failed_refunds = safe_number(
        row,
        "failed_refund_count"
    )

    refund_sla_breaches = safe_number(
        row,
        "refund_sla_breach_count"
    )

    amount_mismatches = safe_number(
        row,
        "refund_amount_mismatch_count"
    )

    delayed_order_refunds = safe_number(
        row,
        "delayed_order_refund_count"
    )


    if pending_refunds > 0:
        score += 3
        reasons.append(
            "Pending refund"
        )

    if failed_refunds > 0:
        score += 4
        reasons.append(
            "Failed refund"
        )

    if refund_sla_breaches > 0:
        score += 3
        reasons.append(
            "Refund SLA breach"
        )

    if amount_mismatches > 0:
        score += 2
        reasons.append(
            "Refund amount mismatch"
        )

    if delayed_order_refunds > 0:
        score += 2
        reasons.append(
            "Refund linked to delayed order"
        )



    negative_complaints = safe_number(
        row,
        "negative_complaint_count"
    )

    angry_complaints = safe_number(
        row,
        "angry_complaint_count"
    )

    complaint_count = safe_number(
        row,
        "complaint_count"
    )


    if negative_complaints > 0:
        score += 2
        reasons.append(
            "Negative customer complaint"
        )

    if angry_complaints > 0:
        score += 3
        reasons.append(
            "Angry customer complaint"
        )

    if complaint_count > 1:
        score += 2
        reasons.append(
            "Repeated customer complaints"
        )


    escalated_tickets = safe_number(
        row,
        "escalated_ticket_count"
    )

    ticket_sla_breaches = safe_number(
        row,
        "ticket_sla_breach_count"
    )

    repeated_support = safe_flag(
        row,
        "repeated_support_issue_flag"
    )


    if escalated_tickets > 0:
        score += 4
        reasons.append(
            "Escalated support ticket"
        )

    if ticket_sla_breaches > 0:
        score += 3
        reasons.append(
            "Support ticket SLA breach"
        )

    if repeated_support:
        score += 2
        reasons.append(
            "Repeated support issue"
        )


    customer_segment = str(
        row.get("segment", "")
    ).strip().lower()

    if customer_segment in {
        "premium",
        "vip",
        "high_value"
    }:
        score += 2
        reasons.append(
            "High customer-impact segment"
        )


    if score >= 12:
        priority = "Critical"

    elif score >= 8:
        priority = "High"

    elif score >= 4:
        priority = "Medium"

    else:
        priority = "Low"


    if not reasons:
        reasons.append(
            "No major operational risk signals"
        )


    return pd.Series({
        "issue_score": int(score),
        "issue_priority": priority,
        "priority_reasons": "; ".join(reasons)
    })


priority_results = priority_df.apply(
    classify_issue_priority,
    axis=1
)


priority_df[
    [
        "issue_score",
        "issue_priority",
        "priority_reasons"
    ]
] = priority_results[
    [
        "issue_score",
        "issue_priority",
        "priority_reasons"
    ]
]

final_feature_table = priority_df.copy()


priority_summary = (
    final_feature_table[
        "issue_priority"
    ]
    .value_counts()
    .reindex(
        [
            "Critical",
            "High",
            "Medium",
            "Low"
        ],
        fill_value=0
    )
    .rename_axis("issue_priority")
    .reset_index(
        name="order_count"
    )
)

priority_summary["percentage"] = (
    priority_summary["order_count"]
    / len(final_feature_table)
    * 100
).round(2)


print("PRIORITY SUMMARY")


display(priority_summary)

print("\nPRIORITY CLASSIFICATION PREVIEW")

priority_preview_columns = [
    "order_id",
    "customer_id",
    "seller_id",
    "delivery_delay_days",
    "delay_bucket",
    "pending_refund_count",
    "failed_refund_count",
    "complaint_count",
    "negative_complaint_count",
    "angry_complaint_count",
    "ticket_count",
    "escalated_ticket_count",
    "issue_score",
    "issue_priority",
    "priority_reasons"
]

priority_preview_columns = [
    col
    for col in priority_preview_columns
    if col in final_feature_table.columns
]

display(
    final_feature_table[
        priority_preview_columns
    ]
    .sort_values(
        by="issue_score",
        ascending=False
    )
    .head(15)
)

priority_missing_count = (
    final_feature_table["issue_priority"]
    .isna()
    .sum()
)

print("\nPriority validation:")
print(
    "Records without priority:",
    priority_missing_count
)

if priority_missing_count == 0:
    print("Priority coverage: PASS")
else:
    print("Priority coverage: REVIEW")


add_debug_fix(
    section="Section 14",
    issue_found=(
        "Priority classification was not implemented as a multi-signal "
        "business scoring model."
    ),
    fix_applied=(
        "Created a scoring function combining delivery delay, refund risk, "
        "complaint sentiment/repetition, support ticket escalation/SLA risk, "
        "and customer-impact segment signals."
    ),
    validation_output=(
        f"Priority assigned to {len(final_feature_table)} records; "
        f"{priority_missing_count} records have no priority."
    )
)

display(get_debug_log())

PRIORITY SUMMARY


,issue_priority,order_count,percentage
0,Critical,10,0.20
1,High,121,2.43
2,Medium,581,11.66
3,Low,4272,85.71



PRIORITY CLASSIFICATION PREVIEW


,order_id,customer_id,seller_id,delivery_delay_days,delay_bucket,pending_refund_count,failed_refund_count,complaint_count,negative_complaint_count,angry_complaint_count,ticket_count,escalated_ticket_count,issue_score,issue_priority,priority_reasons
3442,ord0003455,cust000338,sell0972,NaN,Delivery Date Missing or Invalid,1,0,0,0,0,2,1,14,Critical,Pending refund; Refund amount mismatch; Escala...
1039,ord0001040,cust000804,sell0396,NaN,Delivery Date Missing or Invalid,0,1,0,0,0,2,1,13,Critical,Failed refund; Escalated support ticket; Suppo...
3284,ord0003297,cust000512,sell0010,NaN,Delivery Date Missing or Invalid,0,1,0,0,0,2,1,13,Critical,Failed refund; Escalated support ticket; Suppo...
3683,ord0003696,cust000395,sell0307,NaN,Delivery Date Missing or Invalid,1,1,0,0,0,1,1,13,Critical,Pending refund; Failed refund; Refund amount m...
2098,ord0002104,cust000831,sell0004,NaN,Delivery Date Missing or Invalid,0,1,0,0,0,2,1,13,Critical,Failed refund; Escalated support ticket; Suppo...
4,ord0000005,cust001168,sell0981,NaN,Delivery Date Missing or Invalid,1,0,0,0,0,3,1,12,Critical,Pending refund; Escalated support ticket; Supp...
970,ord0000971,cust000145,sell0190,NaN,Delivery Date Missing or Invalid,1,0,0,0,0,2,2,12,Critical,Pending refund; Escalated support ticket; Supp...
2354,ord0002361,cust000786,sell0882,NaN,Delivery Date Missing or Invalid,1,0,0,0,0,2,1,12,Critical,Pending refund; Escalated support ticket; Supp...
421,ord0000422,cust000756,sell0957,NaN,Delivery Date Missing or Invalid,1,0,0,0,0,1,1,12,Critical,Pending refund; Refund amount mismatch; Escala...
3376,ord0003389,cust000764,sell0695,NaN,Delivery Date Missing or Invalid,1,0,0,0,0,4,2,12,Critical,Pending refund; Escalated support ticket; Supp...



Priority validation:
Records without priority: 0
Priority coverage: PASS


,section,issue_found,fix_applied,validation_output
0,Section 1,Imports and project configuration needed to be...,Reviewed and centralized the required Pandas a...,Python/Pandas environment started successfully.
1,Section 2,Inherited core loading section required verifi...,Loaded orders.csv from the extracted DATA_DIR ...,orders_df loaded successfully with 5000 rows.
2,Section 3,Order status summary referenced incorrect colu...,Changed the status analysis to use the correct...,"Summary table, duplicate order count, missing-..."
3,Section 5,"Customer and seller filenames were incorrect, ...",Implemented a reusable FlipkartIssueDataLoader...,"Loaded datasets: ['orders', 'customers', 'sell..."
4,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
5,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
6,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
7,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
8,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
9,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...


## Section 15: Recommended Action Generator

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `final_feature_table`

**How to approach:**
- Create a function that reads priority and reasons.
- Return specific support/refund/seller/city actions.
- Add recommended_action to final table.

**Expected output format:**
- Action summary table.
- Preview with order_id, priority, action.

**Hint:**
- Recommendations should help a support lead decide what to do next.

In [ ]:
# SECTION 15 — RECOMMENDED ACTION GENERATOR

if "final_feature_table" not in globals():
    raise RuntimeError(
        "final_feature_table is not available. "
        "Please run Section 13 and Section 14 first."
    )


action_df = final_feature_table.copy()

def generate_recommended_action(row):
    """
    Generate a practical operational action for the order.
    """

    priority = str(
        row.get("issue_priority", "")
    ).strip().lower()

    reasons = str(
        row.get("priority_reasons", "")
    ).strip().lower()

    actions = []


    if priority == "critical":
        actions.append(
            "Immediate support lead review and customer contact"
        )

    elif priority == "high":
        actions.append(
            "Prioritize for same-day support follow-up"
        )

    elif priority == "medium":
        actions.append(
            "Review during the next support operations cycle"
        )

    else:
        actions.append(
            "Continue routine monitoring"
        )

    if "severe delivery delay" in reasons:

        actions.append(
            "Escalate delivery issue and provide an immediate customer update"
        )

    elif "moderate delivery delay" in reasons:

        actions.append(
            "Check shipment status and provide a delivery update"
        )

    elif "minor delivery delay" in reasons:

        actions.append(
            "Monitor delivery progress and update the customer if required"
        )

    elif "delivery promise breached" in reasons:

        actions.append(
            "Check shipment status and resolve the breached delivery promise"
        )

    if "pending refund" in reasons:

        actions.append(
            "Follow up with refund operations on the pending refund"
        )

    if "failed refund" in reasons:

        actions.append(
            "Escalate failed refund for refund-status resolution"
        )

    if "refund sla breach" in reasons:

        actions.append(
            "Prioritize refund SLA breach for immediate resolution"
        )

    if "refund amount mismatch" in reasons:

        actions.append(
            "Verify refund amount against the original order amount"
        )

    if "refund linked to delayed order" in reasons:

        actions.append(
            "Coordinate refund and delivery resolution for the delayed order"
        )

    if "angry customer complaint" in reasons:

        actions.append(
            "Prioritize complaint handling and customer-care follow-up"
        )

    elif "negative customer complaint" in reasons:

        actions.append(
            "Review the complaint and provide customer resolution"
        )

    if "repeated customer complaints" in reasons:

        actions.append(
            "Investigate the repeated complaint pattern"
        )


    if "escalated support ticket" in reasons:

        actions.append(
            "Review the escalated ticket with the assigned support team"
        )

    if "support ticket sla breach" in reasons:

        actions.append(
            "Accelerate resolution of the support ticket SLA breach"
        )

    if "repeated support issue" in reasons:

        actions.append(
            "Investigate the recurring support issue"
        )


    if "high customer-impact segment" in reasons:

        actions.append(
            "Apply enhanced customer-care follow-up"
        )


    unique_actions = []

    for action in actions:

        if action not in unique_actions:
            unique_actions.append(action)


    return "; ".join(unique_actions)


action_df["recommended_action"] = action_df.apply(
    generate_recommended_action,
    axis=1
)

final_feature_table = action_df.copy()

action_summary = (
    final_feature_table
    .groupby(
        "issue_priority",
        dropna=False
    )
    .agg(
        order_count=(
            "order_id",
            "nunique"
        )
    )
    .reset_index()
)

priority_order = {
    "Critical": 1,
    "High": 2,
    "Medium": 3,
    "Low": 4
}

action_summary["priority_order"] = (
    action_summary["issue_priority"]
    .map(priority_order)
)

action_summary = (
    action_summary
    .sort_values("priority_order")
    .drop(columns="priority_order")
    .reset_index(drop=True)
)

print("RECOMMENDED ACTION SUMMARY")

display(action_summary)


print("\nORDER-LEVEL RECOMMENDED ACTION PREVIEW")

action_preview_columns = [
    "order_id",
    "issue_priority",
    "issue_score",
    "priority_reasons",
    "recommended_action"
]

action_preview_columns = [
    column
    for column in action_preview_columns
    if column in final_feature_table.columns
]

display(
    final_feature_table[
        action_preview_columns
    ]
    .sort_values(
        by="issue_score",
        ascending=False
    )
    .head(15)
)


missing_action_count = (
    final_feature_table[
        "recommended_action"
    ]
    .isna()
    .sum()
)

empty_action_count = (
    final_feature_table[
        "recommended_action"
    ]
    .astype("string")
    .str.strip()
    .eq("")
    .sum()
)


print("\nACTION VALIDATION")
print(
    "Missing recommended actions:",
    missing_action_count
)
print(
    "Empty recommended actions:",
    empty_action_count
)

if (
    missing_action_count == 0
    and empty_action_count == 0
):
    print("Recommended action coverage: PASS")
else:
    print("Recommended action coverage: REVIEW")

add_debug_fix(
    section="Section 15",
    issue_found=(
        "Recommended actions were not available as a clear next-step "
        "operation for each prioritized case."
    ),
    fix_applied=(
        "Implemented a rule-based recommended action generator using "
        "priority and operational reasons to create support, delivery, "
        "refund, complaint, and ticket actions."
    ),
    validation_output=(
        f"Recommended actions generated for {len(final_feature_table)} "
        f"orders; {missing_action_count} missing and "
        f"{empty_action_count} empty actions."
    )
)

display(get_debug_log())

RECOMMENDED ACTION SUMMARY


,issue_priority,order_count
0,Critical,10
1,High,121
2,Medium,581
3,Low,4272



ORDER-LEVEL RECOMMENDED ACTION PREVIEW


,order_id,issue_priority,issue_score,priority_reasons,recommended_action
3442,ord0003455,Critical,14,Pending refund; Refund amount mismatch; Escala...,Immediate support lead review and customer con...
1039,ord0001040,Critical,13,Failed refund; Escalated support ticket; Suppo...,Immediate support lead review and customer con...
3284,ord0003297,Critical,13,Failed refund; Escalated support ticket; Suppo...,Immediate support lead review and customer con...
3683,ord0003696,Critical,13,Pending refund; Failed refund; Refund amount m...,Immediate support lead review and customer con...
2098,ord0002104,Critical,13,Failed refund; Escalated support ticket; Suppo...,Immediate support lead review and customer con...
4,ord0000005,Critical,12,Pending refund; Escalated support ticket; Supp...,Immediate support lead review and customer con...
970,ord0000971,Critical,12,Pending refund; Escalated support ticket; Supp...,Immediate support lead review and customer con...
2354,ord0002361,Critical,12,Pending refund; Escalated support ticket; Supp...,Immediate support lead review and customer con...
421,ord0000422,Critical,12,Pending refund; Refund amount mismatch; Escala...,Immediate support lead review and customer con...
3376,ord0003389,Critical,12,Pending refund; Escalated support ticket; Supp...,Immediate support lead review and customer con...



ACTION VALIDATION
Missing recommended actions: 0
Empty recommended actions: 0
Recommended action coverage: PASS


,section,issue_found,fix_applied,validation_output
0,Section 1,Imports and project configuration needed to be...,Reviewed and centralized the required Pandas a...,Python/Pandas environment started successfully.
1,Section 2,Inherited core loading section required verifi...,Loaded orders.csv from the extracted DATA_DIR ...,orders_df loaded successfully with 5000 rows.
2,Section 3,Order status summary referenced incorrect colu...,Changed the status analysis to use the correct...,"Summary table, duplicate order count, missing-..."
3,Section 5,"Customer and seller filenames were incorrect, ...",Implemented a reusable FlipkartIssueDataLoader...,"Loaded datasets: ['orders', 'customers', 'sell..."
4,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
5,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
6,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
7,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
8,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
9,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...


## Section 16: AI-Ready Prompt Generator

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `final_feature_table`

**How to approach:**
- Create structured prompts from case data.
- Include only available facts.
- Do not call external APIs.
- Add prompt preview column.

**Expected output format:**
- Preview table with order_id and ai_prompt_preview.

**Hint:**
- This is prompt generation only, not AI integration.

In [ ]:
# SECTION 16 — AI-READY PROMPT GENERATOR

if "final_feature_table" not in globals():
    raise RuntimeError(
        "final_feature_table is not available. "
        "Please run Sections 13–15 first."
    )

ai_prompt_df = final_feature_table.copy()


def prompt_value(row, column, default="Not available"):
    """
    Return a clean value from the current row.

    Only values that are actually available in the
    final feature table are used.
    """

    if column not in row.index:
        return default

    value = row[column]

    if pd.isna(value):
        return default

    value = str(value).strip()

    if value == "":
        return default

    return value

def generate_ai_prompt(row):
    """
    Generate an AI-ready prompt from the available order-level
    case facts.

    IMPORTANT:
    This function only creates text.
    It does not call any external AI API.
    """

    order_id = prompt_value(
        row,
        "order_id"
    )

    customer_id = prompt_value(
        row,
        "customer_id"
    )

    seller_id = prompt_value(
        row,
        "seller_id"
    )

    delivery_city = prompt_value(
        row,
        "delivery_city"
    )

    order_status = prompt_value(
        row,
        "order_status_clean"
    )

    order_amount = prompt_value(
        row,
        "order_amount"
    )

    delay_days = prompt_value(
        row,
        "delivery_delay_days"
    )

    delay_bucket = prompt_value(
        row,
        "delay_bucket"
    )

    refund_count = prompt_value(
        row,
        "refund_count",
        "0"
    )

    pending_refunds = prompt_value(
        row,
        "pending_refund_count",
        "0"
    )

    failed_refunds = prompt_value(
        row,
        "failed_refund_count",
        "0"
    )

    refund_sla_breaches = prompt_value(
        row,
        "refund_sla_breach_count",
        "0"
    )

    complaint_count = prompt_value(
        row,
        "complaint_count",
        "0"
    )

    negative_complaints = prompt_value(
        row,
        "negative_complaint_count",
        "0"
    )

    angry_complaints = prompt_value(
        row,
        "angry_complaint_count",
        "0"
    )

    ticket_count = prompt_value(
        row,
        "ticket_count",
        "0"
    )

    escalated_tickets = prompt_value(
        row,
        "escalated_ticket_count",
        "0"
    )

    ticket_sla_breaches = prompt_value(
        row,
        "ticket_sla_breach_count",
        "0"
    )

    issue_score = prompt_value(
        row,
        "issue_score",
        "0"
    )

    issue_priority = prompt_value(
        row,
        "issue_priority"
    )

    priority_reasons = prompt_value(
        row,
        "priority_reasons"
    )

    recommended_action = prompt_value(
        row,
        "recommended_action"
    )

    prompt = f"""
You are assisting a customer support operations team.

Review the following synthetic order issue case using only the
facts provided below. Do not invent missing information.

CASE IDENTIFICATION

Order ID: {order_id}
Customer ID: {customer_id}
Seller ID: {seller_id}
Delivery City: {delivery_city}

ORDER DETAILS

Order Amount: {order_amount}
Order Status: {order_status}

DELIVERY DETAILS

Delivery Delay Days: {delay_days}
Delay Bucket: {delay_bucket}

REFUND DETAILS

Refund Records: {refund_count}
Pending Refunds: {pending_refunds}
Failed Refunds: {failed_refunds}
Refund SLA Breaches: {refund_sla_breaches}

COMPLAINT DETAILS

Complaint Count: {complaint_count}
Negative Complaints: {negative_complaints}
Angry Complaints: {angry_complaints}

SUPPORT TICKET DETAILS

Ticket Count: {ticket_count}
Escalated Tickets: {escalated_tickets}
Ticket SLA Breaches: {ticket_sla_breaches}

PRIORITY DETAILS

Issue Score: {issue_score}
Issue Priority: {issue_priority}
Priority Reasons: {priority_reasons}

CURRENT RECOMMENDED ACTION

{recommended_action}

REQUEST

1. Summarize the main issue in this case.
2. Identify the most important operational risk signals.
3. Suggest the next support or operations steps.
4. Keep the response concise and action-oriented.
5. Use only the facts provided above.
6. Do not assume information that is not available.
""".strip()

    return prompt


ai_prompt_df["ai_prompt_preview"] = (
    ai_prompt_df.apply(
        generate_ai_prompt,
        axis=1
    )
)

final_feature_table = ai_prompt_df.copy()

ai_prompt_preview = final_feature_table[
    [
        "order_id",
        "ai_prompt_preview"
    ]
].head(10)

print("AI-READY PROMPT PREVIEW")

display(ai_prompt_preview)


missing_prompt_count = (
    final_feature_table[
        "ai_prompt_preview"
    ]
    .isna()
    .sum()
)

empty_prompt_count = (
    final_feature_table[
        "ai_prompt_preview"
    ]
    .astype("string")
    .str.strip()
    .eq("")
    .sum()
)

print("\nAI PROMPT VALIDATION")
print(
    "Missing prompts:",
    missing_prompt_count
)

print(
    "Empty prompts:",
    empty_prompt_count
)

if (
    missing_prompt_count == 0
    and empty_prompt_count == 0
):
    print("AI-ready prompt coverage: PASS")
else:
    print("AI-ready prompt coverage: REVIEW")


print("\nAI usage note:")
print(
    "AI-ready prompts were generated using Python rules. "
    "No external AI API or AI model was called."
)


add_debug_fix(
    section="Section 16",
    issue_found=(
        "AI-ready prompt generation was incomplete."
    ),
    fix_applied=(
        "Created a structured Python-based prompt generator using "
        "available order, delivery, refund, complaint, ticket, "
        "priority, and recommended-action facts."
    ),
    validation_output=(
        f"Prompts generated for {len(final_feature_table)} orders; "
        f"{missing_prompt_count} missing and "
        f"{empty_prompt_count} empty prompts."
    )
)

display(get_debug_log())

AI-READY PROMPT PREVIEW


,order_id,ai_prompt_preview
0,ord0000001,You are assisting a customer support operation...
1,ord0000002,You are assisting a customer support operation...
2,ord0000003,You are assisting a customer support operation...
3,ord0000004,You are assisting a customer support operation...
4,ord0000005,You are assisting a customer support operation...
5,ord0000006,You are assisting a customer support operation...
6,ord0000007,You are assisting a customer support operation...
7,ord0000008,You are assisting a customer support operation...
8,ord0000009,You are assisting a customer support operation...
9,ord0000010,You are assisting a customer support operation...



AI PROMPT VALIDATION
Missing prompts: 0
Empty prompts: 0
AI-ready prompt coverage: PASS

AI usage note:
AI-ready prompts were generated using Python rules. No external AI API or AI model was called.


,section,issue_found,fix_applied,validation_output
0,Section 1,Imports and project configuration needed to be...,Reviewed and centralized the required Pandas a...,Python/Pandas environment started successfully.
1,Section 2,Inherited core loading section required verifi...,Loaded orders.csv from the extracted DATA_DIR ...,orders_df loaded successfully with 5000 rows.
2,Section 3,Order status summary referenced incorrect colu...,Changed the status analysis to use the correct...,"Summary table, duplicate order count, missing-..."
3,Section 5,"Customer and seller filenames were incorrect, ...",Implemented a reusable FlipkartIssueDataLoader...,"Loaded datasets: ['orders', 'customers', 'sell..."
4,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
5,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
6,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
7,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
8,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
9,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...


## Section 17: Colab GUI/Search/Filter Experience

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `final_feature_table`

**How to approach:**
- Use ipywidgets or equivalent.
- Allow filtering by priority, city, and ID search.
- Display a compact case table.

**Expected output format:**
- Interactive widget or fallback output.
- Filtered case preview.

**Hint:**
- The GUI should make the notebook feel like an internal tool.

In [ ]:
# SECTION 17 — COLAB GUI / SEARCH / FILTER EXPERIENCE

if "final_feature_table" not in globals():
    raise RuntimeError(
        "final_feature_table is not available. "
        "Please run Sections 13–16 first."
    )

gui_df = final_feature_table.copy()

try:

    import ipywidgets as widgets
    from IPython.display import display, clear_output

    WIDGETS_AVAILABLE = True

except ImportError:

    WIDGETS_AVAILABLE = False

search_columns = [
    "order_id",
    "customer_id",
    "delivery_city",
    "issue_priority"
]

for column in search_columns:

    if column in gui_df.columns:

        gui_df[column] = (
            gui_df[column]
            .astype("string")
            .fillna("")
            .str.strip()
        )

compact_columns = [
    "order_id",
    "customer_id",
    "seller_id",
    "delivery_city",
    "delay_bucket",
    "issue_score",
    "issue_priority",
    "recommended_action"
]

compact_columns = [
    column
    for column in compact_columns
    if column in gui_df.columns
]


def filter_cases(priority="All",
                 city="All",
                 search_text=""):
    """
    Filter the final feature table by priority, city,
    order ID, or customer ID.
    """

    filtered_df = gui_df.copy()


    if (
        priority != "All"
        and "issue_priority" in filtered_df.columns
    ):

        filtered_df = filtered_df[
            filtered_df["issue_priority"] == priority
        ]


    if (
        city != "All"
        and "delivery_city" in filtered_df.columns
    ):

        filtered_df = filtered_df[
            filtered_df["delivery_city"] == city
        ]

    search_text = (
        str(search_text)
        .strip()
        .lower()
    )

    if search_text:

        order_match = (
            filtered_df["order_id"]
            .str.lower()
            .str.contains(
                search_text,
                na=False
            )
        )

        customer_match = (
            filtered_df["customer_id"]
            .str.lower()
            .str.contains(
                search_text,
                na=False
            )
        )

        filtered_df = filtered_df[
            order_match | customer_match
        ]


    if "issue_score" in filtered_df.columns:

        filtered_df = filtered_df.sort_values(
            by="issue_score",
            ascending=False
        )


    return filtered_df


if WIDGETS_AVAILABLE:

    priority_values = ["All"]

    if "issue_priority" in gui_df.columns:

        priorities = (
            gui_df["issue_priority"]
            .dropna()
            .unique()
            .tolist()
        )

        priorities = sorted(
            [str(value) for value in priorities]
        )

        priority_values.extend(
            priorities
        )


    priority_dropdown = widgets.Dropdown(
        options=priority_values,
        value="All",
        description="Priority:",
        layout=widgets.Layout(
            width="250px"
        )
    )


    city_values = ["All"]

    if "delivery_city" in gui_df.columns:

        cities = (
            gui_df["delivery_city"]
            .dropna()
            .unique()
            .tolist()
        )

        cities = sorted(
            [str(value) for value in cities]
        )

        city_values.extend(
            cities
        )


    city_dropdown = widgets.Dropdown(
        options=city_values,
        value="All",
        description="City:",
        layout=widgets.Layout(
            width="300px"
        )
    )


    search_box = widgets.Text(
        value="",
        placeholder="Order ID or Customer ID",
        description="Search:",
        layout=widgets.Layout(
            width="400px"
        )
    )


    search_button = widgets.Button(
        description="Search Cases",
        button_style="primary",
        layout=widgets.Layout(
            width="180px"
        )
    )


    clear_button = widgets.Button(
        description="Clear Filters",
        layout=widgets.Layout(
            width="180px"
        )
    )

    output_area = widgets.Output()


    def run_case_search(button=None):

        with output_area:

            clear_output(wait=True)

            filtered_cases = filter_cases(
                priority=priority_dropdown.value,
                city=city_dropdown.value,
                search_text=search_box.value
            )


            print("ORDER ISSUE RESOLUTION SEARCH")


            print(
                "Matching cases:",
                len(filtered_cases)
            )


            if filtered_cases.empty:

                print(
                    "\nNo matching cases found."
                )

            else:

                display(
                    filtered_cases[
                        compact_columns
                    ].head(20)
                )


    def clear_filters(button=None):

        priority_dropdown.value = "All"
        city_dropdown.value = "All"
        search_box.value = ""

        run_case_search()


    search_button.on_click(
        run_case_search
    )

    clear_button.on_click(
        clear_filters
    )


    print("ORDER ISSUE RESOLUTION DASHBOARD")


    print(
        "Use the filters below to search for support cases."
    )

    display(
        widgets.HBox([
            priority_dropdown,
            city_dropdown
        ])
    )

    display(
        widgets.HBox([
            search_box,
            search_button,
            clear_button
        ])
    )

    display(output_area)


    run_case_search()


else:

    print(
        "ipywidgets is not available."
    )

    print(
        "Showing fallback case preview instead."
    )

    fallback_cases = filter_cases()

    display(
        fallback_cases[
            compact_columns
        ].head(20)
    )

add_debug_fix(
    section="Section 17",
    issue_found=(
        "The inherited notebook did not provide a usable Colab-compatible "
        "search and filter interface."
    ),
    fix_applied=(
        "Implemented an ipywidgets-based interface with priority and city "
        "filters, order/customer ID search, a clear-filters option, and "
        "a fallback preview when widgets are unavailable."
    ),
    validation_output=(
        "GUI initialized successfully and a compact case preview was displayed."
    )
)

display(get_debug_log())

ORDER ISSUE RESOLUTION DASHBOARD
Use the filters below to search for support cases.


Output()

,section,issue_found,fix_applied,validation_output
0,Section 1,Imports and project configuration needed to be...,Reviewed and centralized the required Pandas a...,Python/Pandas environment started successfully.
1,Section 2,Inherited core loading section required verifi...,Loaded orders.csv from the extracted DATA_DIR ...,orders_df loaded successfully with 5000 rows.
2,Section 3,Order status summary referenced incorrect colu...,Changed the status analysis to use the correct...,"Summary table, duplicate order count, missing-..."
3,Section 5,"Customer and seller filenames were incorrect, ...",Implemented a reusable FlipkartIssueDataLoader...,"Loaded datasets: ['orders', 'customers', 'sell..."
4,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
5,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
6,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
7,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
8,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
9,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...


## Section 18: Final Report Export and Preview

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `final_feature_table`

**How to approach:**
- Select required report schema columns.
- Add generated timestamp.
- Export final CSV.
- Preview top priority cases.

**Expected output format:**
- final_issue_resolution_report.csv.
- Priority summary table.
- Final report preview.

**Hint:**
- The final report should include business signals, not just IDs.

In [ ]:
# SECTION 18 — FINAL REPORT EXPORT AND PREVIEW

from datetime import datetime
from pathlib import Path
import pandas as pd


if "final_feature_table" not in globals():
    raise RuntimeError(
        "final_feature_table is not available. "
        "Please run Sections 13–17 first."
    )


report_df = final_feature_table.copy()


required_report_columns = [
    "order_id",
    "customer_id",
    "seller_id",
    "delivery_city",
    "order_amount",
    "order_status_clean",
    "delivery_delay_days",
    "delay_bucket",
    "refund_count",
    "pending_refund_count",
    "failed_refund_count",
    "complaint_count",
    "ticket_count",
    "issue_score",
    "issue_priority",
    "recommended_action",
    "ai_prompt_preview"
]


missing_report_columns = [
    column
    for column in required_report_columns
    if column not in report_df.columns
]

if missing_report_columns:

    raise RuntimeError(
        "Required report columns are missing: "
        + ", ".join(missing_report_columns)
    )


final_report = report_df[
    required_report_columns
].copy()


final_report["generated_timestamp"] = (
    datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)


REPORT_FILE = "final_issue_resolution_report.csv"

final_report.to_csv(
    REPORT_FILE,
    index=False
)


report_path = Path(REPORT_FILE)

print("FINAL ISSUE-RESOLUTION REPORT")

print(
    "Report exported successfully:",
    report_path.resolve()
)

print(
    "Rows:",
    len(final_report)
)

print(
    "Columns:",
    len(final_report.columns)
)


priority_order = [
    "Critical",
    "High",
    "Medium",
    "Low"
]

priority_summary = (
    final_report[
        "issue_priority"
    ]
    .value_counts()
    .reindex(
        priority_order,
        fill_value=0
    )
    .rename_axis(
        "issue_priority"
    )
    .reset_index(
        name="order_count"
    )
)

priority_summary["percentage"] = (
    priority_summary["order_count"]
    / len(final_report)
    * 100
).round(2)


print("\nPRIORITY SUMMARY")

display(priority_summary)


print("\nTOP PRIORITY CASES")

top_priority_cases = (
    final_report
    .sort_values(
        by="issue_score",
        ascending=False
    )
    .head(20)
)

display(top_priority_cases)


print("\nFINAL REPORT PREVIEW")

display(
    final_report.head(10)
)

missing_schema_after_export = [
    column
    for column in required_report_columns
    if column not in final_report.columns
]

missing_priority_count = (
    final_report["issue_priority"]
    .isna()
    .sum()
)

missing_action_count = (
    final_report["recommended_action"]
    .isna()
    .sum()
)

missing_prompt_count = (
    final_report["ai_prompt_preview"]
    .isna()
    .sum()
)


duplicate_order_count = (
    final_report["order_id"]
    .duplicated()
    .sum()
)


print("\nFINAL REPORT VALIDATION")

print(
    "Missing required columns:",
    len(missing_schema_after_export)
)

print(
    "Missing priority values:",
    missing_priority_count
)

print(
    "Missing recommended actions:",
    missing_action_count
)

print(
    "Missing AI prompt previews:",
    missing_prompt_count
)

print(
    "Duplicate order IDs:",
    duplicate_order_count
)


if (
    len(missing_schema_after_export) == 0
    and missing_priority_count == 0
    and missing_action_count == 0
    and missing_prompt_count == 0
    and duplicate_order_count == 0
):

    print(
        "\nFinal report validation: PASS"
    )

else:

    print(
        "\nFinal report validation: REVIEW"
    )

add_debug_fix(
    section="Section 18",
    issue_found=(
        "The final report required a defined PRD schema, business "
        "signals, export functionality, and validation evidence."
    ),
    fix_applied=(
        "Selected the required report fields, added a generated timestamp, "
        "exported final_issue_resolution_report.csv, displayed priority "
        "summary and top cases, and validated the final report."
    ),
    validation_output=(
        f"Final report contains {len(final_report)} rows and "
        f"{len(final_report.columns)} columns; "
        f"{duplicate_order_count} duplicate order IDs detected."
    )
)

display(get_debug_log())

FINAL ISSUE-RESOLUTION REPORT
Report exported successfully: /content/final_issue_resolution_report.csv
Rows: 4984
Columns: 18

PRIORITY SUMMARY


,issue_priority,order_count,percentage
0,Critical,10,0.20
1,High,121,2.43
2,Medium,581,11.66
3,Low,4272,85.71



TOP PRIORITY CASES


,order_id,customer_id,seller_id,delivery_city,order_amount,order_status_clean,delivery_delay_days,delay_bucket,refund_count,pending_refund_count,failed_refund_count,complaint_count,ticket_count,issue_score,issue_priority,recommended_action,ai_prompt_preview,generated_timestamp
3442,ord0003455,cust000338,sell0972,kolkata,55743.04,delayed,NaN,Delivery Date Missing or Invalid,1,1,0,0,2,14,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
1039,ord0001040,cust000804,sell0396,ahmedabad,14247.12,delivered,NaN,Delivery Date Missing or Invalid,1,0,1,0,2,13,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
3284,ord0003297,cust000512,sell0010,mumbai,17652.60,delivered,NaN,Delivery Date Missing or Invalid,1,0,1,0,2,13,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
3683,ord0003696,cust000395,sell0307,chennai,59530.80,processing,NaN,Delivery Date Missing or Invalid,3,1,1,0,1,13,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
2098,ord0002104,cust000831,sell0004,bengaluru,15832.00,delivered,NaN,Delivery Date Missing or Invalid,1,0,1,0,2,13,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
4,ord0000005,cust001168,sell0981,nagpur,56251.50,cancelled,NaN,Delivery Date Missing or Invalid,1,1,0,0,3,12,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
970,ord0000971,cust000145,sell0190,noida,45062.38,delivered,NaN,Delivery Date Missing or Invalid,1,1,0,0,2,12,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
2354,ord0002361,cust000786,sell0882,lucknow,64394.30,processing,NaN,Delivery Date Missing or Invalid,1,1,0,0,2,12,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
421,ord0000422,cust000756,sell0957,mumbai,46937.94,processing,NaN,Delivery Date Missing or Invalid,2,1,0,0,1,12,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
3376,ord0003389,cust000764,sell0695,kolkata,43425.60,delivered,NaN,Delivery Date Missing or Invalid,1,1,0,0,4,12,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59



FINAL REPORT PREVIEW


,order_id,customer_id,seller_id,delivery_city,order_amount,order_status_clean,delivery_delay_days,delay_bucket,refund_count,pending_refund_count,failed_refund_count,complaint_count,ticket_count,issue_score,issue_priority,recommended_action,ai_prompt_preview,generated_timestamp
0,ord0000001,cust000771,sell0475,delhi,57056.16,delivered,NaN,Delivery Date Missing or Invalid,0,0,0,0,0,0,Low,Continue routine monitoring,You are assisting a customer support operation...,2026-09-10 11:14:59
1,ord0000002,cust000122,sell0035,gurugram,25038.34,delivered,NaN,Delivery Date Missing or Invalid,1,1,0,0,0,3,Low,Continue routine monitoring; Follow up with re...,You are assisting a customer support operation...,2026-09-10 11:14:59
2,ord0000003,cust000784,sell0785,lucknow,29439.12,processing,NaN,Delivery Date Missing or Invalid,0,0,0,0,0,0,Low,Continue routine monitoring,You are assisting a customer support operation...,2026-09-10 11:14:59
3,ord0000004,cust000288,sell0211,indore,60697.51,delivered,NaN,Delivery Date Missing or Invalid,0,0,0,0,0,0,Low,Continue routine monitoring,You are assisting a customer support operation...,2026-09-10 11:14:59
4,ord0000005,cust001168,sell0981,nagpur,56251.50,cancelled,NaN,Delivery Date Missing or Invalid,1,1,0,0,3,12,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
5,ord0000006,cust001093,sell0225,pune,34649.02,delivered,NaN,Delivery Date Missing or Invalid,1,1,0,0,0,3,Low,Continue routine monitoring; Follow up with re...,You are assisting a customer support operation...,2026-09-10 11:14:59
6,ord0000007,cust000829,sell0157,delhi,24879.38,delivered,NaN,Delivery Date Missing or Invalid,0,0,0,0,0,0,Low,Continue routine monitoring,You are assisting a customer support operation...,2026-09-10 11:14:59
7,ord0000008,cust000980,sell0319,chennai,20175.42,cancelled,NaN,Delivery Date Missing or Invalid,1,1,0,0,0,3,Low,Continue routine monitoring; Follow up with re...,You are assisting a customer support operation...,2026-09-10 11:14:59
8,ord0000009,cust001004,sell0719,noida,31226.45,refund pending,NaN,Delivery Date Missing or Invalid,0,0,0,0,0,0,Low,Continue routine monitoring,You are assisting a customer support operation...,2026-09-10 11:14:59
9,ord0000010,custx960572,sell0202,chennai,58844.06,cancelled,NaN,Delivery Date Missing or Invalid,1,0,0,0,0,2,Low,Continue routine monitoring; Verify refund amo...,You are assisting a customer support operation...,2026-09-10 11:14:59



FINAL REPORT VALIDATION
Missing required columns: 0
Missing priority values: 0
Missing recommended actions: 0
Missing AI prompt previews: 0
Duplicate order IDs: 0

Final report validation: PASS


,section,issue_found,fix_applied,validation_output
0,Section 1,Imports and project configuration needed to be...,Reviewed and centralized the required Pandas a...,Python/Pandas environment started successfully.
1,Section 2,Inherited core loading section required verifi...,Loaded orders.csv from the extracted DATA_DIR ...,orders_df loaded successfully with 5000 rows.
2,Section 3,Order status summary referenced incorrect colu...,Changed the status analysis to use the correct...,"Summary table, duplicate order count, missing-..."
3,Section 5,"Customer and seller filenames were incorrect, ...",Implemented a reusable FlipkartIssueDataLoader...,"Loaded datasets: ['orders', 'customers', 'sell..."
4,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
5,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
6,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
7,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
8,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
9,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...


## Section 19: Final Validation Checks

**Developer handover note:** Complete this section as the Python Developer responsible for repairing the inherited internal tool.

**Datasets to use:**
- `final_report`
- `debug_log`
- `notebook outputs`

**How to approach:**
- Check report file exists.
- Check required columns.
- Check priorities/actions/prompts are non-null.
- Check debug log evidence exists.

**Expected output format:**
- final_validation_df with check, passed, details.
- Final success message.

**Hint:**
- Validation output is the final engineering proof before handover.

In [ ]:
# SECTION 19 — FINAL VALIDATION CHECKS

from pathlib import Path
import pandas as pd


final_validation_results = []


def add_final_validation(check, passed, details):
    """
    Add one final validation check.
    """

    final_validation_results.append({
        "check": check,
        "passed": bool(passed),
        "details": details
    })


report_object_exists = (
    "final_report" in globals()
    and isinstance(final_report, pd.DataFrame)
)

add_final_validation(
    "Final report DataFrame exists",
    report_object_exists,
    (
        "final_report exists and is a Pandas DataFrame."
        if report_object_exists
        else
        "final_report is missing or is not a Pandas DataFrame."
    )
)


REPORT_FILE = "final_issue_resolution_report.csv"

report_file_exists = Path(REPORT_FILE).exists()

add_final_validation(
    "Final report CSV exists",
    report_file_exists,
    (
        f"{REPORT_FILE} exists in the current working directory."
        if report_file_exists
        else
        f"{REPORT_FILE} was not found."
    )
)

required_report_columns = [
    "order_id",
    "customer_id",
    "seller_id",
    "delivery_city",
    "order_amount",
    "order_status_clean",
    "delivery_delay_days",
    "delay_bucket",
    "refund_count",
    "pending_refund_count",
    "failed_refund_count",
    "complaint_count",
    "ticket_count",
    "issue_score",
    "issue_priority",
    "recommended_action",
    "ai_prompt_preview"
]


if report_object_exists:

    missing_columns = [
        column
        for column in required_report_columns
        if column not in final_report.columns
    ]

else:

    missing_columns = required_report_columns.copy()


schema_passed = (
    len(missing_columns) == 0
)

add_final_validation(
    "Required report columns",
    schema_passed,
    (
        "All required PRD report columns are present."
        if schema_passed
        else
        f"Missing columns: {missing_columns}"
    )
)


if report_object_exists:

    missing_priority = (
        final_report["issue_priority"]
        .isna()
        .sum()
    )

    empty_priority = (
        final_report["issue_priority"]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
    )

else:

    missing_priority = -1
    empty_priority = -1


priority_passed = (
    report_object_exists
    and missing_priority == 0
    and empty_priority == 0
)

add_final_validation(
    "Priority coverage",
    priority_passed,
    (
        "Every final report record has an issue priority."
        if priority_passed
        else
        f"Missing priority values: {missing_priority}; "
        f"empty priority values: {empty_priority}."
    )
)


if report_object_exists:

    missing_actions = (
        final_report["recommended_action"]
        .isna()
        .sum()
    )

    empty_actions = (
        final_report["recommended_action"]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
    )

else:

    missing_actions = -1
    empty_actions = -1


action_passed = (
    report_object_exists
    and missing_actions == 0
    and empty_actions == 0
)

add_final_validation(
    "Recommended action coverage",
    action_passed,
    (
        "Every final report record has a recommended action."
        if action_passed
        else
        f"Missing actions: {missing_actions}; "
        f"empty actions: {empty_actions}."
    )
)


if report_object_exists:

    missing_prompts = (
        final_report["ai_prompt_preview"]
        .isna()
        .sum()
    )

    empty_prompts = (
        final_report["ai_prompt_preview"]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
    )

else:

    missing_prompts = -1
    empty_prompts = -1


prompt_passed = (
    report_object_exists
    and missing_prompts == 0
    and empty_prompts == 0
)

add_final_validation(
    "AI prompt coverage",
    prompt_passed,
    (
        "Every final report record has an AI-ready prompt preview."
        if prompt_passed
        else
        f"Missing prompts: {missing_prompts}; "
        f"empty prompts: {empty_prompts}."
    )
)


debug_object_exists = (
    "debug_log" in globals()
    and isinstance(debug_log, list)
)

if debug_object_exists:

    debug_log_count = len(debug_log)

else:

    debug_log_count = 0


debug_passed = (
    debug_object_exists
    and debug_log_count > 0
)

add_final_validation(
    "Debug log evidence",
    debug_passed,
    (
        f"Debug log contains {debug_log_count} recorded repair entries."
        if debug_passed
        else
        "Debug log is missing or contains no repair entries."
    )
)


if report_object_exists:

    duplicate_order_ids = (
        final_report["order_id"]
        .duplicated()
        .sum()
    )

else:

    duplicate_order_ids = -1


order_uniqueness_passed = (
    report_object_exists
    and duplicate_order_ids == 0
)

add_final_validation(
    "Final order uniqueness",
    order_uniqueness_passed,
    (
        "No duplicate order IDs are present in the final report."
        if order_uniqueness_passed
        else
        f"Duplicate order ID rows: {duplicate_order_ids}."
    )
)


if (
    report_object_exists
    and "final_feature_table" in globals()
):

    feature_rows = len(final_feature_table)
    report_rows = len(final_report)

    row_count_passed = (
        feature_rows == report_rows
    )

else:

    feature_rows = -1
    report_rows = -1
    row_count_passed = False


add_final_validation(
    "Final report row-count consistency",
    row_count_passed,
    (
        f"Final feature table rows ({feature_rows}) "
        f"match final report rows ({report_rows})."
        if row_count_passed
        else
        f"Feature table rows: {feature_rows}; "
        f"report rows: {report_rows}."
    )
)


if report_object_exists:

    allowed_priorities = {
        "Critical",
        "High",
        "Medium",
        "Low"
    }

    invalid_priority_values = sorted(
        set(
            final_report["issue_priority"]
            .dropna()
            .astype(str)
        )
        - allowed_priorities
    )

else:

    invalid_priority_values = []


priority_label_passed = (
    report_object_exists
    and len(invalid_priority_values) == 0
)

add_final_validation(
    "Priority label validity",
    priority_label_passed,
    (
        "All priority values are valid."
        if priority_label_passed
        else
        f"Invalid priority values: {invalid_priority_values}"
    )
)


final_validation_df = pd.DataFrame(
    final_validation_results,
    columns=[
        "check",
        "passed",
        "details"
    ]
)


print("FINAL ENGINEERING VALIDATION")

display(final_validation_df)


failed_checks = final_validation_df[
    final_validation_df["passed"] == False
]

passed_checks = final_validation_df[
    final_validation_df["passed"] == True
]

print("\nValidation results:")
print(
    "Passed checks:",
    len(passed_checks)
)

print(
    "Failed checks:",
    len(failed_checks)
)


if len(failed_checks) == 0:

    print("FINAL HANDOVER STATUS: READY")

    print(
        "All final engineering validation checks passed."
    )

    print(
        "The notebook is ready for handover."
    )

else:

    print("FINAL HANDOVER STATUS: REVIEW REQUIRED")

    print(
        "One or more final validation checks failed."
    )

    print("\nChecks requiring attention:")

    display(
        failed_checks
    )

add_debug_fix(
    section="Section 19",
    issue_found=(
        "Final engineering validation was required before handover "
        "to confirm report integrity, schema completeness, output "
        "coverage, and developer evidence."
    ),
    fix_applied=(
        "Implemented final checks for report existence, required "
        "columns, priority/action/prompt completeness, debug-log "
        "evidence, duplicate order IDs, row-count consistency, "
        "and valid priority labels."
    ),
    validation_output=(
        f"{len(passed_checks)} checks passed and "
        f"{len(failed_checks)} checks failed."
    )
)

display(get_debug_log())

FINAL ENGINEERING VALIDATION


,check,passed,details
0,Final report DataFrame exists,True,final_report exists and is a Pandas DataFrame.
1,Final report CSV exists,True,final_issue_resolution_report.csv exists in th...
2,Required report columns,True,All required PRD report columns are present.
3,Priority coverage,True,Every final report record has an issue priority.
4,Recommended action coverage,True,Every final report record has a recommended ac...
5,AI prompt coverage,True,Every final report record has an AI-ready prom...
6,Debug log evidence,True,Debug log contains 33 recorded repair entries.
7,Final order uniqueness,True,No duplicate order IDs are present in the fina...
8,Final report row-count consistency,True,Final feature table rows (4984) match final re...
9,Priority label validity,True,All priority values are valid.



Validation results:
Passed checks: 10
Failed checks: 0
FINAL HANDOVER STATUS: READY
All final engineering validation checks passed.
The notebook is ready for handover.


,section,issue_found,fix_applied,validation_output
0,Section 1,Imports and project configuration needed to be...,Reviewed and centralized the required Pandas a...,Python/Pandas environment started successfully.
1,Section 2,Inherited core loading section required verifi...,Loaded orders.csv from the extracted DATA_DIR ...,orders_df loaded successfully with 5000 rows.
2,Section 3,Order status summary referenced incorrect colu...,Changed the status analysis to use the correct...,"Summary table, duplicate order count, missing-..."
3,Section 5,"Customer and seller filenames were incorrect, ...",Implemented a reusable FlipkartIssueDataLoader...,"Loaded datasets: ['orders', 'customers', 'sell..."
4,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
5,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
6,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
7,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
8,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
9,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...


## Notebook-Based Deliverable Placeholders

Fill the following sections inside this same notebook. Do not submit these as separate files.

In [ ]:
# Deliverable Placeholders

if "debug_log" in globals() and len(debug_log) > 0:

    debug_fix_log = pd.DataFrame(
        debug_log,
        columns=[
            "section",
            "issue_found",
            "fix_applied",
            "validation_output"
        ]
    )

else:

    debug_fix_log = pd.DataFrame(
        columns=[
            "section",
            "issue_found",
            "fix_applied",
            "validation_output"
        ]
    )


print("DEBUG FIX LOG")
display(debug_fix_log)

DEBUG FIX LOG


,section,issue_found,fix_applied,validation_output
0,Section 1,Imports and project configuration needed to be...,Reviewed and centralized the required Pandas a...,Python/Pandas environment started successfully.
1,Section 2,Inherited core loading section required verifi...,Loaded orders.csv from the extracted DATA_DIR ...,orders_df loaded successfully with 5000 rows.
2,Section 3,Order status summary referenced incorrect colu...,Changed the status analysis to use the correct...,"Summary table, duplicate order count, missing-..."
3,Section 5,"Customer and seller filenames were incorrect, ...",Implemented a reusable FlipkartIssueDataLoader...,"Loaded datasets: ['orders', 'customers', 'sell..."
4,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
5,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
6,Section 6,The inherited validation logic used incorrect ...,Corrected the validation fields to match the l...,16 validation checks completed; 9 checks requi...
7,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
8,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...
9,Section 7,The inherited notebook repeated cleaning logic...,"Created reusable helper functions for text, da...",6 datasets cleaned successfully; cleaned previ...


## AI Prompt Usage Log

Document how AI helped you, if used. You must still validate and explain all final code and outputs yourself.

In [ ]:
# AI PROMPT USAGE LOG

ai_prompt_usage_log = pd.DataFrame([
    {
        "task": "Notebook structure and section guidance",
        "prompt_used": "Help complete the inherited notebook sections according to the supplied PRD and handover requirements.",
        "ai_output_used": "Used suggested implementation structure and coding guidance as a starting point.",
        "human_validation_done": "Yes — code was reviewed, adapted to the actual dataset schema, and executed."
    },
    {
        "task": "Debugging inherited code",
        "prompt_used": "Identify and fix runtime errors such as incorrect column names and incomplete logic.",
        "ai_output_used": "Used debugging suggestions to correct field names, dependencies, and processing logic.",
        "human_validation_done": "Yes — errors were reproduced, fixes were applied, and affected sections were re-run."
    },
    {
        "task": "Data validation and relationship checks",
        "prompt_used": "Implement validation for required columns, duplicate IDs, orphan relationships, invalid dates, and suspicious amounts.",
        "ai_output_used": "Used the proposed validation logic and adapted it to the actual dataset fields.",
        "human_validation_done": "Yes — validation tables and outputs were checked in the notebook."
    },
    {
        "task": "Priority classification",
        "prompt_used": "Create a multi-signal priority scoring function using delivery, refund, complaint, ticket, and customer-impact signals.",
        "ai_output_used": "Used the scoring approach as implementation guidance.",
        "human_validation_done": "Yes — priority coverage and resulting classifications were executed and checked."
    },
    {
        "task": "Recommended action generation",
        "prompt_used": "Generate actionable support/refund/delivery/ticket recommendations from issue priority and reasons.",
        "ai_output_used": "Used the rule-based action-generation approach.",
        "human_validation_done": "Yes — recommended actions were generated for the final feature table and validated."
    },
    {
        "task": "AI-ready prompt generation",
        "prompt_used": "Create structured case prompts using only available order-level facts without calling an external AI API.",
        "ai_output_used": "Used the prompt template and Python-based prompt-generation logic.",
        "human_validation_done": "Yes — prompt coverage was checked and no external AI API was called."
    }
])


print("AI PROMPT USAGE LOG")
display(ai_prompt_usage_log)

AI PROMPT USAGE LOG


,task,prompt_used,ai_output_used,human_validation_done
0,Notebook structure and section guidance,Help complete the inherited notebook sections ...,Used suggested implementation structure and co...,"Yes — code was reviewed, adapted to the actual..."
1,Debugging inherited code,Identify and fix runtime errors such as incorr...,Used debugging suggestions to correct field na...,"Yes — errors were reproduced, fixes were appli..."
2,Data validation and relationship checks,"Implement validation for required columns, dup...",Used the proposed validation logic and adapted...,Yes — validation tables and outputs were check...
3,Priority classification,Create a multi-signal priority scoring functio...,Used the scoring approach as implementation gu...,Yes — priority coverage and resulting classifi...
4,Recommended action generation,Generate actionable support/refund/delivery/ti...,Used the rule-based action-generation approach.,Yes — recommended actions were generated for t...
5,AI-ready prompt generation,Create structured case prompts using only avai...,Used the prompt template and Python-based prom...,Yes — prompt coverage was checked and no exter...


## PRD Completion Mapping

Map product requirements to notebook sections and evidence.

In [ ]:
# PRD COMPLETION MAPPING

prd_completion_mapping = pd.DataFrame([
    {
        "prd_requirement": "FR-01 — Load dataset ZIP in Colab",
        "notebook_section": "Dataset Setup / Section 2",
        "evidence": "Dataset ZIP upload/extraction code and DATA_DIR creation.",
        "completion_status": "Completed"
    },
    {
        "prd_requirement": "FR-02 — Load all data files",
        "notebook_section": "Section 5",
        "evidence": "FlipkartIssueDataLoader loads CSV, Excel, JSON, and TXT files.",
        "completion_status": "Completed"
    },
    {
        "prd_requirement": "FR-03 — Validate data integrity",
        "notebook_section": "Section 6",
        "evidence": "Required-column, duplicate-ID, relationship, date, and amount validation tables.",
        "completion_status": "Completed"
    },
    {
        "prd_requirement": "FR-04 — Clean and standardize data",
        "notebook_section": "Section 7",
        "evidence": "Reusable text, date, and numeric cleaning functions with cleaned previews.",
        "completion_status": "Completed"
    },
    {
        "prd_requirement": "FR-05 — Analyze order delays",
        "notebook_section": "Section 8",
        "evidence": "delivery_delay_days, delay_bucket, delay summary, and city delay summary.",
        "completion_status": "Completed"
    },
    {
        "prd_requirement": "FR-06 — Analyze refunds",
        "notebook_section": "Section 9",
        "evidence": "Refund age, pending/failed flags, SLA breach, amount mismatch, and refund priority candidate logic.",
        "completion_status": "Completed"
    },
    {
        "prd_requirement": "FR-07 — Link complaints",
        "notebook_section": "Section 10",
        "evidence": "Order-level complaint aggregation, repeated complaints, negative/angry signals, and complaint type summary.",
        "completion_status": "Completed"
    },
    {
        "prd_requirement": "FR-08 — Map support tickets",
        "notebook_section": "Section 11",
        "evidence": "Ticket status standardization, escalation flags, SLA breach logic, tickets_order, and ticket_team_summary.",
        "completion_status": "Completed"
    },
    {
        "prd_requirement": "FR-09 — Detect duplicate/repeat issues",
        "notebook_section": "Section 12",
        "evidence": "Duplicate order IDs, duplicate-looking order patterns, repeated complaint customers, and repeated support issues.",
        "completion_status": "Completed"
    },
    {
        "prd_requirement": "FR-10 — Build final feature table",
        "notebook_section": "Section 13",
        "evidence": "Aggregated refund/complaint/ticket features merged with customer/seller masters and merge validation.",
        "completion_status": "Completed"
    },
    {
        "prd_requirement": "FR-11 — Classify priority",
        "notebook_section": "Section 14",
        "evidence": "Multi-signal issue_score, issue_priority, and priority_reasons.",
        "completion_status": "Completed"
    },
    {
        "prd_requirement": "FR-12 — Generate recommended action",
        "notebook_section": "Section 15",
        "evidence": "generate_recommended_action() and recommended_action column with action summary.",
        "completion_status": "Completed"
    },
    {
        "prd_requirement": "FR-13 — Generate AI-ready prompts",
        "notebook_section": "Section 16",
        "evidence": "generate_ai_prompt() and ai_prompt_preview; no external AI API call.",
        "completion_status": "Completed"
    },
    {
        "prd_requirement": "FR-14 — Build Colab GUI",
        "notebook_section": "Section 17",
        "evidence": "ipywidgets priority/city filters, order/customer search, and compact case preview.",
        "completion_status": "Completed"
    },
    {
        "prd_requirement": "FR-15 — Export final report",
        "notebook_section": "Section 18",
        "evidence": "final_issue_resolution_report.csv, priority summary, and final report preview.",
        "completion_status": "Completed"
    },
    {
        "prd_requirement": "FR-16 — Run validation checks",
        "notebook_section": "Section 19",
        "evidence": "final_validation_df with report, schema, priority, action, prompt, debug-log, uniqueness, and row-count checks.",
        "completion_status": "Completed"
    },
    {
        "prd_requirement": "Debug Fix Log",
        "notebook_section": "Section 4 + Deliverable Placeholder",
        "evidence": "Structured debug_log with section, issue_found, fix_applied, and validation_output.",
        "completion_status": "Completed"
    },
    {
        "prd_requirement": "AI Prompt Usage Log",
        "notebook_section": "Deliverable Placeholder",
        "evidence": "AI assistance tasks, output usage, and human validation documentation.",
        "completion_status": "Completed"
    }
])


print("PRD COMPLETION MAPPING")

display(prd_completion_mapping)


total_requirements = len(prd_completion_mapping)

completed_requirements = (
    prd_completion_mapping[
        "completion_status"
    ]
    .eq("Completed")
    .sum()
)

print("\nPRD Mapping Summary:")
print("Total mapped requirements:", total_requirements)
print("Completed requirements:", completed_requirements)

if total_requirements == completed_requirements:
    print("Overall PRD mapping status: COMPLETE")
else:
    print("Overall PRD mapping status: REVIEW")

PRD COMPLETION MAPPING


,prd_requirement,notebook_section,evidence,completion_status
0,FR-01 — Load dataset ZIP in Colab,Dataset Setup / Section 2,Dataset ZIP upload/extraction code and DATA_DI...,Completed
1,FR-02 — Load all data files,Section 5,"FlipkartIssueDataLoader loads CSV, Excel, JSON...",Completed
2,FR-03 — Validate data integrity,Section 6,"Required-column, duplicate-ID, relationship, d...",Completed
3,FR-04 — Clean and standardize data,Section 7,"Reusable text, date, and numeric cleaning func...",Completed
4,FR-05 — Analyze order delays,Section 8,"delivery_delay_days, delay_bucket, delay summa...",Completed
5,FR-06 — Analyze refunds,Section 9,"Refund age, pending/failed flags, SLA breach, ...",Completed
6,FR-07 — Link complaints,Section 10,"Order-level complaint aggregation, repeated co...",Completed
7,FR-08 — Map support tickets,Section 11,"Ticket status standardization, escalation flag...",Completed
8,FR-09 — Detect duplicate/repeat issues,Section 12,"Duplicate order IDs, duplicate-looking order p...",Completed
9,FR-10 — Build final feature table,Section 13,Aggregated refund/complaint/ticket features me...,Completed



PRD Mapping Summary:
Total mapped requirements: 18
Completed requirements: 18
Overall PRD mapping status: COMPLETE


## Assumption and Limitation Log

Document assumptions, edge cases, and implementation limitations.

In [ ]:
# ASSUMPTION AND LIMITATION LOG

assumption_limitation_log = pd.DataFrame([
    {
        "type": "Assumption",
        "description": (
            "The project uses synthetic training data only and does not "
            "represent real Flipkart operational records."
        ),
        "impact": (
            "Results are suitable for learning and prototype evaluation, "
            "not for production business decisions."
        ),
        "mitigation": (
            "Keep the notebook clearly documented as a simulated internal "
            "operations prototype."
        )
    },
    {
        "type": "Assumption",
        "description": (
            "The final feature table is maintained at one row per order."
        ),
        "impact": (
            "Order-level reporting and priority classification remain "
            "consistent."
        ),
        "mitigation": (
            "Aggregate refunds, complaints, and tickets by order_id before "
            "merging and validate row counts after merges."
        )
    },
    {
        "type": "Assumption",
        "description": (
            "Orders with a missing actual delivery date may represent "
            "pending shipments."
        ),
        "impact": (
            "Pending orders can still be identified when their promised "
            "delivery date has already passed."
        ),
        "mitigation": (
            "Compare the promised delivery date with the reference date "
            "when actual delivery is unavailable."
        )
    },
    {
        "type": "Edge Case",
        "description": (
            "Missing or invalid delivery dates."
        ),
        "impact": (
            "Delay calculation may not be possible for the affected order."
        ),
        "mitigation": (
            "Use safe date coercion and classify affected records as "
            "'Delivery Date Missing or Invalid' instead of deleting them."
        )
    },
    {
        "type": "Edge Case",
        "description": (
            "Duplicate order IDs and duplicate-looking order records."
        ),
        "impact": (
            "Duplicate records can distort order counts and final reporting."
        ),
        "mitigation": (
            "Flag duplicate IDs and duplicate-looking patterns for review "
            "and maintain one order-level record in the final feature table."
        )
    },
    {
        "type": "Edge Case",
        "description": (
            "Refund amounts may be negative or greater than the linked "
            "order amount."
        ),
        "impact": (
            "Refund totals and refund-priority decisions may be unreliable."
        ),
        "mitigation": (
            "Flag suspicious refund amounts during validation and compare "
            "refund_amount with the linked order_amount."
        )
    },
    {
        "type": "Edge Case",
        "description": (
            "Complaint or support ticket records may reference an order "
            "that does not exist in orders.csv."
        ),
        "impact": (
            "These orphan records cannot be safely linked to an order."
        ),
        "mitigation": (
            "Report orphan relationships during validation instead of "
            "silently deleting the records."
        )
    },
    {
        "type": "Edge Case",
        "description": (
            "Escalated support tickets may have missing resolution time."
        ),
        "impact": (
            "SLA duration cannot be calculated directly."
        ),
        "mitigation": (
            "Flag escalated tickets with missing resolution time for "
            "operational review."
        )
    },
    {
        "type": "Limitation",
        "description": (
            "Priority classification is rule-based and does not use "
            "machine learning."
        ),
        "impact": (
            "Priority results depend on the defined business rules and "
            "weights."
        ),
        "mitigation": (
            "Keep the score explainable by storing issue_score and "
            "priority_reasons for every order."
        )
    },
    {
        "type": "Limitation",
        "description": (
            "AI functionality is limited to generating AI-ready prompts."
        ),
        "impact": (
            "The notebook does not generate an AI response or provide "
            "real-time model recommendations."
        ),
        "mitigation": (
            "Store structured ai_prompt_preview text and explicitly "
            "document that no external AI API is called."
        )
    },
    {
        "type": "Limitation",
        "description": (
            "The GUI is implemented with ipywidgets inside Google Colab "
            "rather than as a production web application."
        ),
        "impact": (
            "The interface is suitable for the notebook prototype but not "
            "for production deployment."
        ),
        "mitigation": (
            "Keep the interface simple, provide fallback output when "
            "widgets are unavailable, and remain within Colab scope."
        )
    },
    {
        "type": "Limitation",
        "description": (
            "The project does not use external APIs, SQL databases, "
            "cloud deployment, or production authentication."
        ),
        "impact": (
            "The tool cannot operate as a connected production system."
        ),
        "mitigation": (
            "Keep all data processing, analysis, and reporting inside "
            "the Colab notebook."
        )
    }
])


print("ASSUMPTION AND LIMITATION LOG")

display(assumption_limitation_log)

print("\nTotal documented items:",
      len(assumption_limitation_log))

print(
    "Assumptions:",
    (assumption_limitation_log["type"] == "Assumption").sum()
)

print(
    "Edge cases:",
    (assumption_limitation_log["type"] == "Edge Case").sum()
)

print(
    "Limitations:",
    (assumption_limitation_log["type"] == "Limitation").sum()
)

ASSUMPTION AND LIMITATION LOG


,type,description,impact,mitigation
0,Assumption,The project uses synthetic training data only ...,Results are suitable for learning and prototyp...,Keep the notebook clearly documented as a simu...
1,Assumption,The final feature table is maintained at one r...,Order-level reporting and priority classificat...,"Aggregate refunds, complaints, and tickets by ..."
2,Assumption,Orders with a missing actual delivery date may...,Pending orders can still be identified when th...,Compare the promised delivery date with the re...
3,Edge Case,Missing or invalid delivery dates.,Delay calculation may not be possible for the ...,Use safe date coercion and classify affected r...
4,Edge Case,Duplicate order IDs and duplicate-looking orde...,Duplicate records can distort order counts and...,Flag duplicate IDs and duplicate-looking patte...
5,Edge Case,Refund amounts may be negative or greater than...,Refund totals and refund-priority decisions ma...,Flag suspicious refund amounts during validati...
6,Edge Case,Complaint or support ticket records may refere...,These orphan records cannot be safely linked t...,Report orphan relationships during validation ...
7,Edge Case,Escalated support tickets may have missing res...,SLA duration cannot be calculated directly.,Flag escalated tickets with missing resolution...
8,Limitation,Priority classification is rule-based and does...,Priority results depend on the defined busines...,Keep the score explainable by storing issue_sc...
9,Limitation,AI functionality is limited to generating AI-r...,The notebook does not generate an AI response ...,Store structured ai_prompt_preview text and ex...



Total documented items: 12
Assumptions: 3
Edge cases: 5
Limitations: 4


## Final Report Preview Evidence

Show report file name, shape, schema, and high-priority preview.

In [ ]:
# FINAL REPORT PREVIEW EVIDENCE

from pathlib import Path


if "final_report" not in globals():
    raise RuntimeError(
        "final_report is not available. "
        "Please run Section 18 — Final Report Export and Preview first."
    )

REPORT_FILE = "final_issue_resolution_report.csv"
report_path = Path(REPORT_FILE)


final_report_preview_evidence = {
    "report_file_name": REPORT_FILE,
    "report_file_path": str(report_path.resolve()),
    "report_file_exists": report_path.exists(),
    "report_shape": final_report.shape,
    "report_row_count": len(final_report),
    "report_column_count": len(final_report.columns),
    "report_schema": list(final_report.columns)
}


print("FINAL REPORT PREVIEW EVIDENCE")


print("\nReport file name:")
print(final_report_preview_evidence["report_file_name"])

print("\nReport file path:")
print(final_report_preview_evidence["report_file_path"])

print("\nReport file exists:")
print(final_report_preview_evidence["report_file_exists"])

print("\nReport shape:")
print(final_report_preview_evidence["report_shape"])

print("\nReport schema:")
for column in final_report.columns:
    print(" -", column)

print("\nHIGH-PRIORITY CASE PREVIEW")

high_priority_preview = (
    final_report[
        final_report["issue_priority"].isin(
            ["Critical", "High"]
        )
    ]
    .sort_values(
        by="issue_score",
        ascending=False
    )
    .head(20)
)

display(high_priority_preview)


if high_priority_preview.empty:

    print(
        "No Critical or High priority cases were found."
    )

    print(
        "Showing the highest-scoring cases instead:"
    )

    high_priority_preview = (
        final_report
        .sort_values(
            by="issue_score",
            ascending=False
        )
        .head(20)
    )

    display(high_priority_preview)


final_report_evidence_summary = pd.DataFrame([
    {
        "evidence_item": "Report File",
        "value": REPORT_FILE
    },
    {
        "evidence_item": "File Exists",
        "value": report_path.exists()
    },
    {
        "evidence_item": "Rows",
        "value": final_report.shape[0]
    },
    {
        "evidence_item": "Columns",
        "value": final_report.shape[1]
    },
    {
        "evidence_item": "High/Critical Cases",
        "value": len(
            final_report[
                final_report["issue_priority"].isin(
                    ["Critical", "High"]
                )
            ]
        )
    }
])

print("\nREPORT EVIDENCE SUMMARY")

display(final_report_evidence_summary)

FINAL REPORT PREVIEW EVIDENCE

Report file name:
final_issue_resolution_report.csv

Report file path:
/content/final_issue_resolution_report.csv

Report file exists:
True

Report shape:
(4984, 18)

Report schema:
 - order_id
 - customer_id
 - seller_id
 - delivery_city
 - order_amount
 - order_status_clean
 - delivery_delay_days
 - delay_bucket
 - refund_count
 - pending_refund_count
 - failed_refund_count
 - complaint_count
 - ticket_count
 - issue_score
 - issue_priority
 - recommended_action
 - ai_prompt_preview
 - generated_timestamp

HIGH-PRIORITY CASE PREVIEW


,order_id,customer_id,seller_id,delivery_city,order_amount,order_status_clean,delivery_delay_days,delay_bucket,refund_count,pending_refund_count,failed_refund_count,complaint_count,ticket_count,issue_score,issue_priority,recommended_action,ai_prompt_preview,generated_timestamp
3442,ord0003455,cust000338,sell0972,kolkata,55743.04,delayed,NaN,Delivery Date Missing or Invalid,1,1,0,0,2,14,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
1039,ord0001040,cust000804,sell0396,ahmedabad,14247.12,delivered,NaN,Delivery Date Missing or Invalid,1,0,1,0,2,13,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
3284,ord0003297,cust000512,sell0010,mumbai,17652.60,delivered,NaN,Delivery Date Missing or Invalid,1,0,1,0,2,13,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
3683,ord0003696,cust000395,sell0307,chennai,59530.80,processing,NaN,Delivery Date Missing or Invalid,3,1,1,0,1,13,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
2098,ord0002104,cust000831,sell0004,bengaluru,15832.00,delivered,NaN,Delivery Date Missing or Invalid,1,0,1,0,2,13,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
4,ord0000005,cust001168,sell0981,nagpur,56251.50,cancelled,NaN,Delivery Date Missing or Invalid,1,1,0,0,3,12,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
2354,ord0002361,cust000786,sell0882,lucknow,64394.30,processing,NaN,Delivery Date Missing or Invalid,1,1,0,0,2,12,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
3376,ord0003389,cust000764,sell0695,kolkata,43425.60,delivered,NaN,Delivery Date Missing or Invalid,1,1,0,0,4,12,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
421,ord0000422,cust000756,sell0957,mumbai,46937.94,processing,NaN,Delivery Date Missing or Invalid,2,1,0,0,1,12,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59
970,ord0000971,cust000145,sell0190,noida,45062.38,delivered,NaN,Delivery Date Missing or Invalid,1,1,0,0,2,12,Critical,Immediate support lead review and customer con...,You are assisting a customer support operation...,2026-09-10 11:14:59



REPORT EVIDENCE SUMMARY


,evidence_item,value
0,Report File,final_issue_resolution_report.csv
1,File Exists,True
2,Rows,4984
3,Columns,18
4,High/Critical Cases,131


## Final Product Walkthrough

Explain the completed internal tool as if handing it back to Product and Engineering.

In [ ]:
# FINAL PRODUCT WALKTHROUGH


final_product_walkthrough = [
    "1. Dataset Loading: The notebook uploads/extracts the synthetic dataset package and loads orders, customers, sellers, refunds, complaints, support tickets, and operational notes using reusable file-loading logic.",

    "2. Data Validation and Cleaning: The tool checks required columns, duplicate IDs, orphan relationships, invalid dates, and suspicious amounts, then standardizes text, dates, and numeric fields without silently deleting invalid records.",

    "3. Delivery and Refund Analysis: Orders are classified into delay buckets, including pending promise-date breaches, while refund records are analyzed for pending/failed status, refund age, SLA breaches, amount mismatches, and delayed-order linkage.",

    "4. Complaint and Ticket Analysis: Customer complaints and support tickets are aggregated at order level so repeated complaints, negative/angry sentiment, escalations, SLA risks, and recurring support issues can be identified safely.",

    "5. Issue Prioritization: The final feature table combines multiple operational signals and assigns an issue score and priority level of Critical, High, Medium, or Low. Priority reasons are retained so support users can understand why a case was prioritized.",

    "6. Recommended Actions: Each prioritized case receives a practical next-step recommendation covering customer support, delivery, refund operations, complaint handling, or support-ticket follow-up.",

    "7. Search and AI-Ready Assistance: The Colab GUI allows users to filter by priority and city and search by order/customer ID. The notebook also generates structured AI-ready prompts from available case facts without calling an external AI API.",

    "8. Final Reporting and Handover: The tool exports final_issue_resolution_report.csv, previews high-priority cases, validates the final report and schema, and keeps debug, AI usage, PRD mapping, assumptions, and validation evidence inside the notebook."
]


print("FINAL PRODUCT WALKTHROUGH")

for point in final_product_walkthrough:
    print("\n" + point)

FINAL PRODUCT WALKTHROUGH

1. Dataset Loading: The notebook uploads/extracts the synthetic dataset package and loads orders, customers, sellers, refunds, complaints, support tickets, and operational notes using reusable file-loading logic.

2. Data Validation and Cleaning: The tool checks required columns, duplicate IDs, orphan relationships, invalid dates, and suspicious amounts, then standardizes text, dates, and numeric fields without silently deleting invalid records.

3. Delivery and Refund Analysis: Orders are classified into delay buckets, including pending promise-date breaches, while refund records are analyzed for pending/failed status, refund age, SLA breaches, amount mismatches, and delayed-order linkage.

4. Complaint and Ticket Analysis: Customer complaints and support tickets are aggregated at order level so repeated complaints, negative/angry sentiment, escalations, SLA risks, and recurring support issues can be identified safely.

5. Issue Prioritization: The final fea

## Final Self-Check

Use this as your final readiness check before submission.

In [ ]:
# FINAL SELF-CHECK

final_self_check_results = []


def add_self_check(check_item, status, notes):
    final_self_check_results.append({
        "check_item": check_item,
        "status": status,
        "notes": notes
    })


datasets_available = (
    "all_data" in globals()
    and isinstance(all_data, dict)
    and len(all_data) >= 6
)

add_self_check(
    "All required datasets loaded",
    "PASS" if datasets_available else "REVIEW",
    (
        f"{len(all_data)} datasets are available."
        if datasets_available
        else
        "Required dataset collection is not available."
    )
)


feature_table_ready = (
    "final_feature_table" in globals()
    and isinstance(final_feature_table, pd.DataFrame)
    and len(final_feature_table) > 0
)

add_self_check(
    "Final feature table created",
    "PASS" if feature_table_ready else "REVIEW",
    (
        f"Final feature table contains {len(final_feature_table)} rows."
        if feature_table_ready
        else
        "final_feature_table is missing or empty."
    )
)


priority_ready = (
    feature_table_ready
    and "issue_score" in final_feature_table.columns
    and "issue_priority" in final_feature_table.columns
    and final_feature_table["issue_priority"].notna().all()
)

add_self_check(
    "Priority classification completed",
    "PASS" if priority_ready else "REVIEW",
    (
        "Issue score and priority are available for all final records."
        if priority_ready
        else
        "Priority score/labels are incomplete."
    )
)


action_ready = (
    feature_table_ready
    and "recommended_action" in final_feature_table.columns
    and final_feature_table["recommended_action"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .all()
)

add_self_check(
    "Recommended actions completed",
    "PASS" if action_ready else "REVIEW",
    (
        "Every final record has a recommended action."
        if action_ready
        else
        "Some records are missing recommended actions."
    )
)


prompt_ready = (
    feature_table_ready
    and "ai_prompt_preview" in final_feature_table.columns
    and final_feature_table["ai_prompt_preview"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .all()
)

add_self_check(
    "AI-ready prompt generation completed",
    "PASS" if prompt_ready else "REVIEW",
    (
        "AI-ready prompts are available for all final records."
        if prompt_ready
        else
        "Some AI-ready prompt previews are missing."
    )
)


report_ready = (
    "final_report" in globals()
    and isinstance(final_report, pd.DataFrame)
    and len(final_report) > 0
)

add_self_check(
    "Final report generated",
    "PASS" if report_ready else "REVIEW",
    (
        f"Final report contains {len(final_report)} rows."
        if report_ready
        else
        "final_report is missing or empty."
    )
)


report_file = Path(
    "final_issue_resolution_report.csv"
)

report_export_ready = report_file.exists()

add_self_check(
    "Final CSV exported",
    "PASS" if report_export_ready else "REVIEW",
    (
        f"File exists: {report_file.resolve()}"
        if report_export_ready
        else
        "Final report CSV was not found."
    )
)


required_report_columns = [
    "order_id",
    "customer_id",
    "seller_id",
    "delivery_city",
    "order_amount",
    "order_status_clean",
    "delivery_delay_days",
    "delay_bucket",
    "refund_count",
    "pending_refund_count",
    "failed_refund_count",
    "complaint_count",
    "ticket_count",
    "issue_score",
    "issue_priority",
    "recommended_action",
    "ai_prompt_preview"
]


if report_ready:

    missing_columns = [
        column
        for column in required_report_columns
        if column not in final_report.columns
    ]

else:

    missing_columns = required_report_columns


schema_ready = len(missing_columns) == 0

add_self_check(
    "Final report schema complete",
    "PASS" if schema_ready else "REVIEW",
    (
        "All PRD-required report columns are present."
        if schema_ready
        else
        f"Missing columns: {missing_columns}"
    )
)


if report_ready:

    duplicate_orders = (
        final_report["order_id"]
        .duplicated()
        .sum()
    )

else:

    duplicate_orders = -1


one_row_per_order = (
    report_ready
    and duplicate_orders == 0
)

add_self_check(
    "Final report contains one row per order",
    "PASS" if one_row_per_order else "REVIEW",
    (
        "No duplicate order IDs were found."
        if one_row_per_order
        else
        f"Duplicate order ID rows: {duplicate_orders}"
    )
)


debug_ready = (
    "debug_log" in globals()
    and isinstance(debug_log, list)
    and len(debug_log) > 0
)

add_self_check(
    "Debug fix log completed",
    "PASS" if debug_ready else "REVIEW",
    (
        f"{len(debug_log)} debug entries recorded."
        if debug_ready
        else
        "Debug log is missing or empty."
    )
)


ai_usage_ready = (
    "ai_prompt_usage_log" in globals()
    and isinstance(ai_prompt_usage_log, pd.DataFrame)
    and len(ai_prompt_usage_log) > 0
)

add_self_check(
    "AI prompt usage log completed",
    "PASS" if ai_usage_ready else "REVIEW",
    (
        f"{len(ai_prompt_usage_log)} AI usage entries documented."
        if ai_usage_ready
        else
        "AI prompt usage log is missing or empty."
    )
)


prd_mapping_ready = (
    "prd_completion_mapping" in globals()
    and isinstance(prd_completion_mapping, pd.DataFrame)
    and len(prd_completion_mapping) > 0
)

if prd_mapping_ready:

    incomplete_prd_items = (
        prd_completion_mapping[
            prd_completion_mapping["completion_status"]
            != "Completed"
        ]
    )

else:

    incomplete_prd_items = pd.DataFrame()


prd_ready = (
    prd_mapping_ready
    and len(incomplete_prd_items) == 0
)

add_self_check(
    "PRD completion mapping completed",
    "PASS" if prd_ready else "REVIEW",
    (
        "All mapped PRD requirements are marked Completed."
        if prd_ready
        else
        f"{len(incomplete_prd_items)} PRD items require review."
    )
)


assumption_log_ready = (
    "assumption_limitation_log" in globals()
    and isinstance(
        assumption_limitation_log,
        pd.DataFrame
    )
    and len(assumption_limitation_log) > 0
)

add_self_check(
    "Assumption and limitation log completed",
    "PASS" if assumption_log_ready else "REVIEW",
    (
        f"{len(assumption_limitation_log)} assumptions, "
        "edge cases, and limitations documented."
        if assumption_log_ready
        else
        "Assumption/limitation log is missing or empty."
    )
)


final_validation_ready = (
    "final_validation_df" in globals()
    and isinstance(
        final_validation_df,
        pd.DataFrame
    )
    and len(final_validation_df) > 0
    and final_validation_df["passed"].all()
)

add_self_check(
    "Final engineering validation passed",
    "PASS" if final_validation_ready else "REVIEW",
    (
        "All final engineering validation checks passed."
        if final_validation_ready
        else
        "One or more final engineering validation checks require review."
    )
)


final_self_check = pd.DataFrame(
    final_self_check_results,
    columns=[
        "check_item",
        "status",
        "notes"
    ]
)


print("FINAL SELF-CHECK")

display(final_self_check)


failed_self_checks = final_self_check[
    final_self_check["status"] != "PASS"
]

passed_self_checks = final_self_check[
    final_self_check["status"] == "PASS"
]


print("\nSelf-check results:")
print(
    "Passed:",
    len(passed_self_checks)
)

print(
    "Requires review:",
    len(failed_self_checks)
)


if len(failed_self_checks) == 0:

    print("FINAL SUBMISSION STATUS: READY")


    print(
        "The notebook has passed the final self-check "
        "and is ready for submission."
    )

else:

    print("FINAL SUBMISSION STATUS: REVIEW REQUIRED")


    print(
        "Please review the failed self-check items below:"
    )

    display(failed_self_checks)

FINAL SELF-CHECK


,check_item,status,notes
0,All required datasets loaded,PASS,7 datasets are available.
1,Final feature table created,PASS,Final feature table contains 4984 rows.
2,Priority classification completed,PASS,Issue score and priority are available for all...
3,Recommended actions completed,PASS,Every final record has a recommended action.
4,AI-ready prompt generation completed,PASS,AI-ready prompts are available for all final r...
5,Final report generated,PASS,Final report contains 4984 rows.
6,Final CSV exported,PASS,File exists: /content/final_issue_resolution_r...
7,Final report schema complete,PASS,All PRD-required report columns are present.
8,Final report contains one row per order,PASS,No duplicate order IDs were found.
9,Debug fix log completed,PASS,34 debug entries recorded.



Self-check results:
Passed: 14
Requires review: 0
FINAL SUBMISSION STATUS: READY
The notebook has passed the final self-check and is ready for submission.


## Presentation and Explanation Notes

Prepare a short explanation for reviewers/stakeholders.

In [ ]:
# PRESENTATION AND EXPLANATION NOTES


presentation_explanation_notes = """
1. Product Overview
The Flipkart Order Issue Resolution Dashboard is a Colab-based
internal operations prototype designed to help support and
marketplace operations teams identify and prioritize order issues
using synthetic operational datasets.

2. Data Processing
The notebook loads orders, customers, sellers, refunds, complaints,
support tickets, and operational notes. Data is validated and cleaned
before analysis, with invalid records flagged for review rather than
silently deleted.

3. Issue Analysis
The tool identifies delayed orders, refund-risk cases, repeated
complaints, escalated support tickets, and other operational signals.
Refunds, complaints, and tickets are aggregated at order level before
being merged into the final feature table.

4. Priority Classification
Multiple business signals are combined to calculate an issue score
and assign Critical, High, Medium, or Low priority. Priority reasons
are stored so support users can understand why a case was prioritized.

5. Recommended Actions
Each case receives a practical recommended action covering support,
delivery, refund, complaint, or ticket follow-up so that a support
lead can decide what to do next.

6. Search and AI-Ready Assistance
The Colab GUI provides priority/city filters and order/customer ID
search. The notebook also generates structured AI-ready prompts from
available case facts. No external AI API is called.

7. Final Reporting
The completed tool exports final_issue_resolution_report.csv,
provides a priority summary and high-priority case preview, and
validates the final report schema and data coverage.

8. Handover Status
The notebook includes the debug fix log, AI prompt usage log,
PRD completion mapping, assumption and limitation log, validation
evidence, walkthrough, and final self-check so Product and Engineering
can review the implementation and handover status.
"""
print("PRESENTATION AND EXPLANATION NOTES")

print(presentation_explanation_notes)

PRESENTATION AND EXPLANATION NOTES

1. Product Overview
The Flipkart Order Issue Resolution Dashboard is a Colab-based
internal operations prototype designed to help support and
marketplace operations teams identify and prioritize order issues
using synthetic operational datasets.

2. Data Processing
The notebook loads orders, customers, sellers, refunds, complaints,
support tickets, and operational notes. Data is validated and cleaned
before analysis, with invalid records flagged for review rather than
silently deleted.

3. Issue Analysis
The tool identifies delayed orders, refund-risk cases, repeated
complaints, escalated support tickets, and other operational signals.
Refunds, complaints, and tickets are aggregated at order level before
being merged into the final feature table.

4. Priority Classification
Multiple business signals are combined to calculate an issue score
and assign Critical, High, Medium, or Low priority. Priority reasons
are stored so support users can understand 

## 100-Mark Evaluation Rubric

| Criteria | Marks |
|---|---:|
| Data loading and file handling for CSV, Excel, JSON, TXT, and ZIP extraction | 8 |
| Data validation and relationship quality checks | 10 |
| Data cleaning and standardization of statuses, dates, text, and amounts | 10 |
| Correct use of Python functions and modular code | 10 |
| OOP implementation and code organization | 8 |
| Core business logic for delays, refunds, complaints, tickets, and priority scoring | 12 |
| Pandas analysis and aggregations for operations insights | 10 |
| Colab GUI/search/filter experience | 8 |
| Report generation and notebook-visible outputs | 7 |
| Debug log, AI usage log, assumptions, and PRD mapping | 7 |
| Presentation & Explanation | 10 |
| **Total** | **100** |

## Final Submission Checklist

Before submitting the completed notebook, confirm:

- [ ] Dataset ZIP was uploaded and extracted successfully.
- [ ] All code cells run from top to bottom in Google Colab.
- [ ] All major section outputs are visible.
- [ ] Debug Fix Log is filled inside the notebook.
- [ ] AI Prompt Usage Log is filled inside the notebook.
- [ ] PRD Completion Mapping is filled inside the notebook.
- [ ] Assumption and Limitation Log is filled inside the notebook.
- [ ] Final Report Preview Evidence is visible.
- [ ] Final Product Walkthrough is complete.
- [ ] Final Self-Check is complete.
- [ ] Presentation and Explanation Notes are complete.
- [ ] The only final deliverable is the completed Colab notebook with all outputs visible.